# 1. Retrieval từ corpus: BM25 + BGE + Qwen + E5

Không dùng submission_17.zip. Nguồn: sáu worker ZIP của structural corpus Stage 8.
Mở notebook ở các phiên riêng: ROLE=bm25 (CPU), bge/qwen/e5 (GPU).
Giữ cùng CAMPAIGN và toàn bộ cấu hình plan. Dense có thể chia SHARD_COUNT/SHARD_INDEX;
BM25 dùng một chỉ mục toàn cục, SHARD_COUNT=1.

Mặc định pilot 20 query / 4 batch, chỉ kiểm tra luồng kỹ thuật. Khi chạy toàn bộ, đổi hai giới hạn
thành None ở tất cả phiên. Mỗi model thực sự encode lại corpus; không tái dùng chỉ mục BGE cũ.
Embedding float32 lưu đủ corpus có thể cần khoảng 22,6 GB/model nếu chiều 1024, ba nhánh khoảng 67,7 GB.
Đặt SAVE_EMBEDDINGS=False để chỉ giữ candidate/checkpoint; vẫn thực hiện embedding thật.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, sys, subprocess, zipfile, shutil, time, threading
from contextlib import contextmanager
ROOT = Path('/content/drive/MyDrive/ViBioMIR')
STAGE8_BUNDLES = ROOT / 'dense_stage8_bge_structure/bundles'
DATA_ZIP = ROOT / 'Data.zip'
QUERY_FILE = ROOT / 'query.parquet'
LINKS_FILE = ROOT / 'Data/official/ViBioMIR/links_corpus_vi_en_zh.parquet'
CAMPAIGN = 'corpus_four_branch_v1'
RUN_PARENT = ROOT / 'corpus_retrieval_trials' / CAMPAIGN
LOCAL = Path('/content/corpus_multiretrieval')
LOCAL.mkdir(parents=True, exist_ok=True)
RUN_PARENT.mkdir(parents=True, exist_ok=True)
os_environ = __import__('os').environ
os_environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
DEBUG_CUDA = False  # Set True in a FRESH session if a CUDA assertion persists.
if DEBUG_CUDA:
    assert 'torch' not in sys.modules, 'Restart session before enabling CUDA diagnostics'
    os_environ['CUDA_LAUNCH_BLOCKING'] = '1'


Mounted at /content/drive


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pyarrow==22.0.0', 'transformers==4.57.1', 'tokenizers==0.22.1',
    'safetensors>=0.4,<1', 'sentencepiece>=0.2,<0.3'], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'pyarrow==22.0.0', 'transformers==4.57.1', 'tokenizers==0.22.1', 'safetensors>=0.4,<1', 'sentencepiece>=0.2,<0.3'], returncode=0)

In [3]:
SUPPORT_FILES = {'qwen_submission_trial.py': '"""Standalone, resumable Qwen scoring of submission evidence and optional subspans.\n\nSource verification here means exact slices of the input submission, not the corpus.\nNo dependency on frozen BGE producer code. No official relevance evaluator included.\n"""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport sqlite3\nimport zipfile\n\nMODEL_ID = \'Qwen/Qwen3-Reranker-0.6B\'\nREVISION = \'6e9e69830b95c52b5fd889b7690dda3329508de3\'\nMODEL_DATE = \'2025-06-09T05:59:29Z\'\nEXPECTED_QUERY_SHA256 = \'ce13508b893bbd6b9169679d540630bc37eb5950cad08b4d5e0e5d7063670c08\'\nEXPECTED_ZIP_SHA256 = \'0963641cb422e4223d6fc4b0c22eb09938f8b03c8227053cc7a398bf33e25dbd\'\nPREFIX = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n          \'and the Instruct provided. Note that the answer can only be "yes" or "no".\'\n          \'<|im_end|>\\n<|im_start|>user\\n\')\nSUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\nINSTRUCTION = \'Given a web search query, retrieve relevant passages that answer the query\'\n\n\ndef require(ok, message):\n    if not ok:\n        raise ValueError(message)\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef signature(value):\n    return digest(canonical(value).encode())\n\n\ndef signed(value):\n    return {**value, \'signature\': signature(value)}\n\n\ndef checked(value):\n    require(isinstance(value, dict) and value.get(\'signature\') ==\n            signature({k: v for k, v in value.items() if k != \'signature\'}), \'Record checksum differs\')\n    return value\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(canonical(value), encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef generation_policy(include_spans=False):\n    return dict(version=\'sentence-windows-v1\', include_spans=include_spans,\n                min_parent_chars=350, min_span_chars=100, sentence_windows=[1, 2, 3],\n                max_subspans_per_parent=4)\n\n\ndef model_policy():\n    return dict(model_id=MODEL_ID, revision=REVISION, commit_date=MODEL_DATE,\n                instruction=INSTRUCTION, max_pair_tokens=4096, truncation=\'reject\',\n                scoring=\'logit_yes_minus_logit_no\', dtype=\'float16\',\n                attention=\'sdpa\', template_version=\'official-transformers-prefix-suffix-v1\')\n\n\ndef sentence_spans(text):\n    """Conservative boundaries; preserve offsets and avoid splitting decimals/abbreviations on \'.\'."""\n    boundaries = [0]\n    for match in re.finditer(r\'\\n+|[!?。！？]+[ \\t]*\', text):\n        boundaries.append(match.end())\n    # Split full stops only before a likely next sentence. Decimals like 0.10 stay intact.\n    for match in re.finditer(r\'\\.(?:[ \\t]+)(?=[A-ZÀ-Ỹ])\', text):\n        boundaries.append(match.end())\n    boundaries = sorted(set(boundaries + [len(text)]))\n    result = []\n    for a, b in zip(boundaries, boundaries[1:]):\n        while a < b and text[a].isspace():\n            a += 1\n        while b > a and text[b-1].isspace():\n            b -= 1\n        if a < b:\n            result.append((a, b))\n    return result\n\n\ndef evenly_choose(values, count):\n    if len(values) <= count:\n        return values\n    if count == 1:\n        return [values[len(values)//2]]\n    return [values[round(i * (len(values)-1)/(count-1))] for i in range(count)]\n\n\ndef validate_generation(policy):\n    require(policy[\'version\'] == \'sentence-windows-v1\' and type(policy[\'include_spans\']) is bool,\n            \'Unsupported generation policy\')\n    require(all(type(policy[k]) is int and policy[k] > 0 for k in\n                (\'min_parent_chars\', \'min_span_chars\', \'max_subspans_per_parent\')), \'Positive span budgets required\')\n    require(policy[\'sentence_windows\'] and all(type(n) is int and n > 0 for n in policy[\'sentence_windows\']),\n            \'Positive sentence-window sizes required\')\n\n\ndef make_candidates(row, policy):\n    validate_generation(policy)\n    candidates = []\n    for index, parent in enumerate(row[\'relevant_chunks\']):\n        text = parent[\'chunk_text\']\n        spans = [(0, len(text), \'original\')]\n        if policy[\'include_spans\'] and len(text) >= policy[\'min_parent_chars\']:\n            sentences = sentence_spans(text)\n            extras = set()\n            for width in policy[\'sentence_windows\']:\n                for start in range(len(sentences)-width+1):\n                    a, b = sentences[start][0], sentences[start+width-1][1]\n                    if b-a >= policy[\'min_span_chars\'] and b-a < len(text):\n                        extras.add((a, b))\n            spans.extend((a, b, \'subspan\') for a, b in evenly_choose(\n                sorted(extras), policy[\'max_subspans_per_parent\']))\n        parent_hash = digest(text.encode())\n        for a, b, kind in spans:\n            value = dict(parent_index=index, parent_text_sha256=parent_hash,\n                         doc_id=parent[\'doc_id\'], start=a, end=b, kind=kind,\n                         text_sha256=digest(text[a:b].encode()))\n            candidates.append({**value, \'candidate_id\': signature(value)})\n    return candidates\n\n\ndef evidence_text(row, candidate):\n    require(type(candidate[\'parent_index\']) is int and 0 <= candidate[\'parent_index\'] < len(row[\'relevant_chunks\']),\n            \'Unknown parent index\')\n    parent = row[\'relevant_chunks\'][candidate[\'parent_index\']]\n    text = parent[\'chunk_text\']\n    a, b = candidate[\'start\'], candidate[\'end\']\n    require(type(a) is int and type(b) is int and 0 <= a < b <= len(text), \'Invalid evidence offsets\')\n    require(parent[\'doc_id\'] == candidate[\'doc_id\'] and digest(text.encode()) == candidate[\'parent_text_sha256\']\n            and digest(text[a:b].encode()) == candidate[\'text_sha256\'], \'Evidence differs from parent submission\')\n    return text[a:b]\n\n\ndef load_inputs(submission_zip, query_file, expected_zip=EXPECTED_ZIP_SHA256,\n                expected_query=EXPECTED_QUERY_SHA256, expected_queries=1200):\n    require(file_hash(submission_zip) == expected_zip, \'Wrong input ZIP; select the original submission_17.zip\')\n    require(file_hash(query_file) == expected_query, \'Official query file differs\')\n    with zipfile.ZipFile(submission_zip) as z:\n        require(z.namelist() == [\'submission.json\'], \'Expected exactly one root submission.json\')\n        raw = z.read(\'submission.json\')\n    rows = json.loads(raw)\n    import pyarrow.parquet as pq\n    queries = pq.read_table(query_file, columns=[\'id\', \'query\']).to_pylist()\n    validate_input_rows(rows, queries, expected_queries)\n    return rows, queries, dict(submission_zip_sha256=expected_zip, submission_json_sha256=digest(raw),\n                               query_sha256=expected_query), raw\n\n\ndef validate_input_rows(rows, queries, expected_queries):\n    require(isinstance(rows, list) and len(rows) == len(queries) == expected_queries, \'Query count differs\')\n    ids = [q[\'id\'] for q in queries]\n    require(all(type(i) is int for i in ids) and len(set(ids)) == len(ids) and [r[\'id\'] for r in rows] == ids,\n            \'Query IDs/order differ or duplicate IDs\')\n    for row, query in zip(rows, queries):\n        require(isinstance(query[\'query\'], str) and query[\'query\'].strip(), \'Empty query\')\n        docs = row[\'relevant_docs\']\n        require(all(type(d) is int for d in docs) and len(set(docs)) == len(docs), \'Invalid doc list\')\n        for i, c in enumerate(row[\'relevant_chunks\']):\n            require(type(c[\'doc_id\']) is int and c[\'doc_id\'] in docs and isinstance(c[\'chunk_text\'], str)\n                    and c[\'chunk_text\'].strip() and type(c[\'chunk_order\']) is int and c[\'chunk_order\'] == i,\n                    \'Invalid parent evidence\')\n\n\ndef run_identity(inputs, generation, policy):\n    validate_generation(generation)\n    require(policy[\'model_id\'] == MODEL_ID and policy[\'revision\'] == REVISION and policy[\'commit_date\'] == MODEL_DATE,\n            \'Use the pinned Qwen checkpoint\')\n    return dict(operation=\'qwen-submission-rescore-v1\', inputs=inputs, generation=generation,\n                model_policy=policy, code_sha256=file_hash(__file__), source_scope=\'input_submission_only\')\n\n\ndef formatted_input(query, passage, policy):\n    body = f"<Instruct>: {policy[\'instruction\']}\\n<Query>: {query}\\n<Document>: {passage}"\n    return PREFIX + body + SUFFIX\n\n\ndef encoded_pair(tokenizer, query, passage, policy):\n    # Exactly the official prefix/body/suffix assembly, with no silent truncation.\n    body = f"<Instruct>: {policy[\'instruction\']}\\n<Query>: {query}\\n<Document>: {passage}"\n    ids = (tokenizer.encode(PREFIX, add_special_tokens=False)\n           + tokenizer.encode(body, add_special_tokens=False)\n           + tokenizer.encode(SUFFIX, add_special_tokens=False))\n    require(len(ids) <= policy[\'max_pair_tokens\'],\n            f"Pair has {len(ids)} tokens > {policy[\'max_pair_tokens\']}; increase policy budget in a new run")\n    return dict(input_ids=ids, attention_mask=[1]*len(ids))\n\n\nclass QwenScorer:\n    def __init__(self, policy, batch_size=8, padded_tokens=8192):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        require(torch.cuda.is_available(), \'Select a CUDA GPU runtime\')\n        require(policy[\'max_pair_tokens\'] <= 32768 and policy[\'max_pair_tokens\'] > 0\n                and policy[\'truncation\'] == \'reject\' and policy[\'scoring\'] == \'logit_yes_minus_logit_no\'\n                and policy[\'dtype\'] == \'float16\' and policy[\'attention\'] == \'sdpa\', \'Unsupported scoring policy\')\n        require(batch_size > 0 and padded_tokens >= policy[\'max_pair_tokens\'], \'Invalid GPU budgets\')\n        self.torch, self.policy = torch, policy\n        self.device = torch.device(\'cuda\')\n        self.batch_size, self.padded_tokens = batch_size, padded_tokens\n        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION,\n                                                       padding_side=\'left\', trust_remote_code=False)\n        self.model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION,\n            torch_dtype=torch.float16, attn_implementation=\'sdpa\', trust_remote_code=False).to(\'cuda\').eval()\n        require(getattr(self.model.config, \'_commit_hash\', None) == REVISION, \'Loaded model revision differs\')\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.yes = self.tokenizer.encode(\'yes\', add_special_tokens=False)\n        self.no = self.tokenizer.encode(\'no\', add_special_tokens=False)\n        require(len(self.yes) == len(self.no) == 1 and self.yes != self.no, \'Unexpected yes/no tokenizer mapping\')\n        count = sum(p.numel() for p in self.model.parameters())\n        require(0 < count <= 15_000_000_000 and policy[\'max_pair_tokens\'] <= self.model.config.max_position_embeddings,\n                \'Model size/context exceeds allowed budget\')\n        self.registry = dict(model_id=MODEL_ID, revision=REVISION, commit_date=MODEL_DATE,\n                             parameters=count, dtype=\'float16\', scoring=policy[\'scoring\'],\n                             transformers=transformers.__version__, torch=torch.__version__,\n                             tokenizer_revision=REVISION, gpu=torch.cuda.get_device_name(0))\n\n    def score(self, query, passages):\n        features = [encoded_pair(self.tokenizer, query, p, self.policy) for p in passages]\n        scores = []\n        index, budget = 0, self.batch_size\n        while index < len(features):\n            n = min(budget, len(features)-index)\n            while n > 1 and max(len(v[\'input_ids\']) for v in features[index:index+n])*n > self.padded_tokens:\n                n -= 1\n            try:\n                result = self._batch_scores(features[index:index+n])\n            except self.torch.cuda.OutOfMemoryError:\n                self.torch.cuda.empty_cache()\n                if n == 1:\n                    raise\n                budget = max(1, n//2)\n                print(f\'CUDA OOM: retry smaller batch={budget}\', flush=True)\n                continue\n            scores.extend(result)\n            index += n\n        return scores\n\n    def _batch_scores(self, features):\n        inputs = self.tokenizer.pad(features, padding=True, return_tensors=\'pt\')\n        inputs = {k: v.to(self.device) for k, v in inputs.items()}\n        with self.torch.inference_mode():\n            # Keep only the last token logits: avoids full B x sequence x vocabulary allocation.\n            logits = self.model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()\n            values = (logits[:, self.yes[0]] - logits[:, self.no[0]]).cpu().tolist()\n        require(all(math.isfinite(s) for s in values), \'Nonfinite Qwen score\')\n        return values\n\n\ndef cache_key(query, text, policy):\n    return signature(dict(query_sha256=digest(query.encode()), text_sha256=digest(text.encode()), policy=policy))\n\n\ndef verify_score_record(record, row, query, candidates, run_sig, policy):\n    checked(record)\n    require(record[\'id\'] == row[\'id\'] == query[\'id\'] and record[\'run_signature\'] == run_sig and\n            record[\'query_sha256\'] == digest(query[\'query\'].encode()) and\n            record[\'candidates_sha256\'] == signature(candidates), \'Score query/input identity differs\')\n    registry = record[\'model_registry\']\n    require(registry[\'model_id\'] == policy[\'model_id\'] and registry[\'revision\'] == policy[\'revision\'] and\n            registry[\'commit_date\'] == policy[\'commit_date\'] and\n            registry[\'tokenizer_revision\'] == policy[\'revision\'] and registry[\'scoring\'] == policy[\'scoring\'] and\n            registry[\'dtype\'] == policy[\'dtype\'] and type(registry[\'parameters\']) is int and\n            0 < registry[\'parameters\'] <= 15_000_000_000, \'Score model registry differs\')\n    require(len(record[\'scores\']) == len(candidates), \'Missing candidate scores\')\n    for c, score in zip(candidates, record[\'scores\']):\n        require(score[\'candidate_id\'] == c[\'candidate_id\'] and type(score[\'score\']) in (int, float)\n                and math.isfinite(score[\'score\']), \'Invalid score rows\')\n    return [v[\'score\'] for v in record[\'scores\']]\n\n\ndef run_scoring(rows, queries, identity, output, local_cache, *, stop_after_queries=5,\n                batch_size=8, padded_tokens=8192, scorer_factory=None):\n    require(stop_after_queries is None or type(stop_after_queries) is int and stop_after_queries > 0,\n            \'Positive pilot limit or None required\')\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    manifest_path = output / \'manifest.json\'\n    run_sig = signature(identity)\n    if manifest_path.exists():\n        previous = checked(read(manifest_path))\n        require(previous[\'identity\'] == identity, \'Scoring inputs changed; use new run directory\')\n    else:\n        require(not any(output.iterdir()), \'Nonempty run without manifest; use new directory\')\n    done, hashes, new_count, scored_pairs = [], {}, 0, 0\n    scorer = None\n    def save(state):\n        write(manifest_path, signed(dict(identity=identity, run_signature=run_sig, state=state,\n            completed_query_ids=done, score_files=hashes, queries=len(rows),\n            new_queries_last_invocation=new_count, scored_unique_pairs_last_invocation=scored_pairs)))\n    save(\'running\')\n    Path(local_cache).parent.mkdir(parents=True, exist_ok=True)\n    with sqlite3.connect(local_cache) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS scores(key TEXT PRIMARY KEY,score REAL NOT NULL)\')\n        try:\n            for row, query in zip(rows, queries):\n                candidates = make_candidates(row, identity[\'generation\'])\n                path = output / f"query-{row[\'id\']}.json"\n                if path.is_file():\n                    verify_score_record(read(path), row, query, candidates, run_sig, identity[\'model_policy\'])\n                else:\n                    keys, texts, missing = [], {}, []\n                    for c in candidates:\n                        text = evidence_text(row, c)\n                        key = cache_key(query[\'query\'], text, identity[\'model_policy\'])\n                        keys.append(key)\n                        texts[key] = text\n                    values = {}\n                    for key in dict.fromkeys(keys):\n                        hit = db.execute(\'SELECT score FROM scores WHERE key=?\', (key,)).fetchone()\n                        if hit:\n                            require(math.isfinite(hit[0]), \'Nonfinite cached score\')\n                            values[key] = hit[0]\n                        else:\n                            missing.append(key)\n                    if scorer is None:\n                        scorer = (scorer_factory or QwenScorer)(identity[\'model_policy\'],\n                            batch_size=batch_size, padded_tokens=padded_tokens)\n                    if missing:\n                        scores = scorer.score(query[\'query\'], [texts[k] for k in missing])\n                        require(len(scores) == len(missing) and all(type(s) in (int, float) and math.isfinite(s) for s in scores),\n                                \'Scorer returned invalid scores\')\n                        for key, score in zip(missing, scores):\n                            db.execute(\'INSERT INTO scores VALUES(?,?)\', (key, float(score)))\n                            values[key] = float(score)\n                        db.commit()\n                        scored_pairs += len(missing)\n                    record = signed(dict(id=row[\'id\'], run_signature=run_sig,\n                        query_sha256=digest(query[\'query\'].encode()), candidates_sha256=signature(candidates),\n                        model_registry=scorer.registry,\n                        scores=[dict(candidate_id=c[\'candidate_id\'], score=values[k]) for c,k in zip(candidates,keys)]))\n                    verify_score_record(record, row, query, candidates, run_sig, identity[\'model_policy\'])\n                    write(path, record)\n                    new_count += 1\n                done.append(row[\'id\']); hashes[str(row[\'id\'])] = file_hash(path)\n                save(\'running\')\n                print(f"Query {row[\'id\']}: {len(done)}/{len(rows)} complete; new this call={new_count}", flush=True)\n                if stop_after_queries is not None and new_count >= stop_after_queries and len(done) < len(rows):\n                    save(\'paused\')\n                    return read(manifest_path)\n        except BaseException:\n            save(\'interrupted\')\n            raise\n    save(\'complete\')\n    return read(manifest_path)\n\n\ndef overlap_iou(a, b):\n    if (a[\'doc_id\'], a[\'parent_index\']) != (b[\'doc_id\'], b[\'parent_index\']):\n        return 0.0  # Source coordinates across different parent chunks are unavailable.\n    overlap = max(0, min(a[\'end\'], b[\'end\']) - max(a[\'start\'], b[\'start\']))\n    return overlap / (max(a[\'end\'], b[\'end\']) - min(a[\'start\'], b[\'start\']))\n\n\ndef select_evidence(row, candidates, scores, limits, originals_only=False):\n    require(type(limits[\'max_chunks\']) is int and limits[\'max_chunks\'] > 0 and\n            type(limits[\'per_doc\']) is int and limits[\'per_doc\'] > 0 and\n            0 < limits[\'overlap_iou\'] <= 1, \'Invalid selection limits\')\n    require(len(candidates) == len(scores) and all(math.isfinite(s) for s in scores), \'Invalid score array\')\n    selected, per_doc, decisions, texts = [], {}, [], {}\n    ranked = sorted(zip(candidates, scores), key=lambda x: (-x[1], x[0][\'candidate_id\']))\n    for c, score in ranked:\n        text = evidence_text(row, c)\n        reason = None\n        if originals_only and c[\'kind\'] != \'original\':\n            reason = \'subspan_disabled\'\n        elif len(selected) >= limits[\'max_chunks\']:\n            reason = \'global_cap\'\n        elif per_doc.get(c[\'doc_id\'], 0) >= limits[\'per_doc\']:\n            reason = \'per_doc_cap\'\n        else:\n            peers = [v for v in selected if v[\'doc_id\'] == c[\'doc_id\']]\n            if any(text == texts[v[\'candidate_id\']] for v in peers):\n                reason = \'same_doc_exact_duplicate\'\n            elif any(overlap_iou(c, v) >= limits[\'overlap_iou\'] for v in peers):\n                reason = \'same_parent_overlap\'\n        if reason is None:\n            selected.append(c)\n            texts[c[\'candidate_id\']] = text\n            per_doc[c[\'doc_id\']] = per_doc.get(c[\'doc_id\'], 0)+1\n        decisions.append(dict(candidate_id=c[\'candidate_id\'], reason=reason or \'selected\', score=score))\n    prediction = dict(id=row[\'id\'], relevant_docs=list(row[\'relevant_docs\']), relevant_chunks=[\n        dict(doc_id=c[\'doc_id\'], chunk_text=texts[c[\'candidate_id\']], chunk_order=i) for i,c in enumerate(selected)])\n    return prediction, selected, decisions\n\n\ndef package_variants(rows, queries, identity, score_dir, destination, baseline_raw, variants):\n    score_dir, destination = Path(score_dir), Path(destination)\n    require(not destination.exists(), \'Choose a new export directory\')\n    require(digest(baseline_raw) == identity[\'inputs\'][\'submission_json_sha256\'], \'Baseline bytes differ\')\n    manifest = checked(read(score_dir / \'manifest.json\'))\n    require(manifest[\'identity\'] == identity and manifest[\'state\'] == \'complete\' and\n            manifest[\'completed_query_ids\'] == [r[\'id\'] for r in rows], \'Finish all scoring queries before export\')\n    run_sig = signature(identity)\n    manifest_hash = file_hash(score_dir / \'manifest.json\')\n    destination.mkdir(parents=True)\n    # S0 is byte-for-byte baseline JSON, repackaged without changing prediction data.\n    with zipfile.ZipFile(destination / \'S0_baseline.zip\', \'w\', zipfile.ZIP_DEFLATED) as z:\n        z.writestr(\'submission.json\', baseline_raw)\n    predictions, counts = {v[\'name\']: [] for v in variants}, {v[\'name\']: [] for v in variants}\n    require(len(predictions) == len(variants) and all(re.fullmatch(r\'[A-Za-z0-9_-]+\', v[\'name\']) for v in variants),\n            \'Unique safe variant names required\')\n    evidence_sidecar = destination / \'EVIDENCE_PROVENANCE.jsonl\'\n    with evidence_sidecar.open(\'w\', encoding=\'utf-8\') as sidecar:\n        for row, query in zip(rows, queries):\n            candidates = make_candidates(row, identity[\'generation\'])\n            path = score_dir / f"query-{row[\'id\']}.json"\n            require(file_hash(path) == manifest[\'score_files\'][str(row[\'id\'])], \'Score file changed after completion\')\n            scores = verify_score_record(read(path), row, query, candidates, run_sig, identity[\'model_policy\'])\n            for variant in variants:\n                pred, selected, decisions = select_evidence(row, candidates, scores, variant[\'limits\'],\n                                                           variant[\'originals_only\'])\n                require(pred[\'relevant_docs\'] == row[\'relevant_docs\'], \'Document list changed\')\n                predictions[variant[\'name\']].append(pred)\n                before = {(c[\'doc_id\'], c[\'chunk_text\']) for c in row[\'relevant_chunks\']}\n                after = {(c[\'doc_id\'], c[\'chunk_text\']) for c in pred[\'relevant_chunks\']}\n                counts[variant[\'name\']].append(dict(id=row[\'id\'], chunks=len(selected),\n                    changed_evidence=before != after, subspans=sum(c[\'kind\']==\'subspan\' for c in selected),\n                    documents_with_chunks=len({c[\'doc_id\'] for c in selected})))\n                sidecar.write(canonical(dict(variant=variant[\'name\'], id=row[\'id\'], selected=selected,\n                                             decisions=decisions))+\'\\n\')\n    require(file_hash(score_dir / \'manifest.json\') == manifest_hash, \'Scoring manifest changed during export\')\n    exports = []\n    for variant in variants:\n        name = variant[\'name\']\n        payload = canonical(predictions[name]).encode()\n        archive = destination / f\'{name}.zip\'\n        with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED) as z:\n            z.writestr(\'submission.json\', payload)\n        with zipfile.ZipFile(archive) as z:\n            require(z.namelist() == [\'submission.json\'] and z.read(\'submission.json\') == payload, \'ZIP differs\')\n        exports.append(dict(variant=variant, zip=archive.name, zip_sha256=file_hash(archive),\n            json_sha256=digest(payload), queries=len(rows),\n            changed_queries=sum(c[\'changed_evidence\'] for c in counts[name]),\n            total_chunks=sum(c[\'chunks\'] for c in counts[name]),\n            total_subspans=sum(c[\'subspans\'] for c in counts[name]), query_statistics=counts[name]))\n    report = dict(validation=\'passed\', validation_scope=\'input submission subset/slices and score integrity\',\n        original_corpus_evidence_verified=False, official_doc_allowlist_verified=False,\n        relevance_measured=False, official_lcs_evaluated=False, source_identity=identity,\n        score_manifest_sha256=manifest_hash, baseline_json_sha256=digest(baseline_raw),\n        provenance_sha256=file_hash(evidence_sidecar), exports=exports)\n    write(destination / \'REPORT.json\', report)\n    return report\n', 'multimodel_evidence.py': '"""Parallel evidence rerankers, rank fusion and optional disagreement arbitration.\n\nAll labels are model judgments. Exact source scope is the input submission only.\n"""\nimport math\nfrom pathlib import Path\nimport sqlite3\nimport zipfile\n\ntry:\n    from . import qwen_submission_trial as core\nexcept ImportError:\n    import qwen_submission_trial as core\n\n\ndef policies():\n    qwen = core.model_policy()\n    return dict(qwen=qwen,\n        bge=dict(model_id=\'BAAI/bge-reranker-v2-m3\',\n            revision=\'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e\', commit_date=\'2024-06-24T14:08:45Z\',\n            max_pair_tokens=4096, truncation=\'reject\', scoring=\'sequence_classification_logit\',\n            dtype=\'float16\', attention=\'sdpa\', template_version=\'tokenizer-pair-v1\'),\n        arbiter={**qwen, \'model_id\': \'Qwen/Qwen3-Reranker-4B\',\n            \'revision\': \'22e683669bc0f0bd69640a1354a6d0aebcfeede5\', \'commit_date\': \'2026-04-16T08:56:07Z\'})\n\n\ndef build_plan(inputs, generation):\n    core.validate_generation(generation)\n    return core.signed(dict(schema=\'multimodel-evidence-v1\', inputs=inputs, generation=generation,\n        models=policies(), code_sha256=core.file_hash(__file__), base_code_sha256=core.file_hash(core.__file__),\n        scope=\'input-submission-evidence-only\', ground_truth=False))\n\n\ndef validate_plan(plan, inputs):\n    core.checked(plan)\n    core.require(plan == build_plan(inputs, plan[\'generation\']), \'Input, model or support code differs from plan\')\n\n\ndef worker_identity(plan, role, shard_index, shard_count, request=None):\n    core.require(role in plan[\'models\'] and type(shard_index) is int and type(shard_count) is int\n        and 1 <= shard_count <= 16 and 0 <= shard_index < shard_count, \'Invalid role/shard assignment\')\n    core.require((role == \'arbiter\') == (request is not None), \'Arbiter needs a verified disagreement request\')\n    return dict(operation=\'multimodel-evidence-worker\', plan_signature=plan[\'signature\'], role=role,\n                shard_index=shard_index, shard_count=shard_count,\n                request_signature=request[\'signature\'] if request else None)\n\n\ndef worker_directory(root, identity):\n    scope = identity[\'request_signature\'] or \'all\'\n    return Path(root) / \'workers\' / identity[\'role\'] / scope / (\n        f"part-{identity[\'shard_index\']}-of-{identity[\'shard_count\']}")\n\n\ndef bge_pair(tokenizer, query, passage, policy):\n    features = dict(tokenizer(query, passage, add_special_tokens=True, truncation=False,\n                              return_token_type_ids=False))\n    core.require(len(features[\'input_ids\']) <= policy[\'max_pair_tokens\'],\n                 f"BGE pair exceeds {policy[\'max_pair_tokens\']} tokens; use a new larger-budget plan")\n    return features\n\n\nclass ModelScorer(core.QwenScorer):\n    """Same score() batching for two architectures; independent model inputs/registries."""\n    def __init__(self, policy, batch_size=8, padded_tokens=8192):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer\n        core.require(policy in policies().values(), \'Use a pinned ensemble model policy\')\n        core.require(torch.cuda.is_available(), \'Select CUDA GPU runtime\')\n        core.require(batch_size > 0 and padded_tokens >= policy[\'max_pair_tokens\'], \'Invalid GPU budgets\')\n        self.torch, self.device, self.policy = torch, torch.device(\'cuda\'), policy\n        self.batch_size, self.padded_tokens = batch_size, padded_tokens\n        self.is_bge = policy[\'model_id\'] == policies()[\'bge\'][\'model_id\']\n        self.tokenizer = AutoTokenizer.from_pretrained(policy[\'model_id\'], revision=policy[\'revision\'],\n            trust_remote_code=False, padding_side=\'right\' if self.is_bge else \'left\')\n        loader = AutoModelForSequenceClassification if self.is_bge else AutoModelForCausalLM\n        self.model = loader.from_pretrained(policy[\'model_id\'], revision=policy[\'revision\'],\n            torch_dtype=torch.float16, attn_implementation=\'sdpa\', trust_remote_code=False,\n            use_safetensors=True).to(self.device).eval()\n        core.require(getattr(self.model.config, \'_commit_hash\', None) == policy[\'revision\'], \'Loaded revision differs\')\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.yes = self.tokenizer.encode(\'yes\', add_special_tokens=False)\n        self.no = self.tokenizer.encode(\'no\', add_special_tokens=False)\n        if self.is_bge:\n            core.require(self.model.config.num_labels == 1, \'BGE requires one classification logit\')\n        else:\n            core.require(len(self.yes) == len(self.no) == 1 and self.yes != self.no, \'Unexpected yes/no token IDs\')\n        count = sum(p.numel() for p in self.model.parameters())\n        context = self.model.config.max_position_embeddings - (2 if self.is_bge else 0)\n        core.require(0 < count <= 15_000_000_000 and policy[\'max_pair_tokens\'] <= context, \'Model/context limit exceeded\')\n        self.registry = dict(model_id=policy[\'model_id\'], revision=policy[\'revision\'],\n            tokenizer_revision=policy[\'revision\'], commit_date=policy[\'commit_date\'], parameters=count,\n            dtype=policy[\'dtype\'], scoring=policy[\'scoring\'], torch=torch.__version__,\n            transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))\n\n    def score(self, query, passages):\n        if not self.is_bge:\n            return super().score(query, passages)\n        features = [bge_pair(self.tokenizer, query, p, self.policy) for p in passages]\n        scores, index, budget = [], 0, self.batch_size\n        while index < len(features):\n            n = min(budget, len(features)-index)\n            while n > 1 and max(len(v[\'input_ids\']) for v in features[index:index+n])*n > self.padded_tokens:\n                n -= 1\n            try:\n                values = self._batch_scores(features[index:index+n])\n            except self.torch.cuda.OutOfMemoryError:\n                self.torch.cuda.empty_cache()\n                if n == 1:\n                    raise\n                budget = max(1, n//2)\n                print(\'CUDA OOM: retry BGE batch\', budget, flush=True)\n                continue\n            scores.extend(values); index += n\n        return scores\n\n    def _batch_scores(self, features):\n        if not self.is_bge:\n            return super()._batch_scores(features)\n        inputs = {k: v.to(self.device) for k,v in self.tokenizer.pad(features,padding=True,return_tensors=\'pt\').items()}\n        with self.torch.inference_mode():\n            values = self.model(**inputs, return_dict=True).logits.reshape(-1).float().cpu().tolist()\n        core.require(len(values) == len(features) and all(math.isfinite(x) for x in values), \'Invalid BGE logits\')\n        return values\n\n\ndef verify_record(record, row, query, candidates, identity, policy):\n    # Compatible score envelope, but worker identity binds the model/plan/shard separately.\n    return core.verify_score_record(record, row, query, candidates, core.signature(identity), policy)\n\n\ndef tasks_for(rows, queries, plan, identity, request=None):\n    lookup = {t[\'id\']: t for t in request[\'tasks\']} if request else None\n    tasks = []\n    for index, (row, query) in enumerate(zip(rows, queries)):\n        if index % identity[\'shard_count\'] != identity[\'shard_index\']:\n            continue\n        if lookup is not None and row[\'id\'] not in lookup:\n            continue\n        candidates = core.make_candidates(row, plan[\'generation\'])\n        if lookup is not None:\n            t = lookup[row[\'id\']]\n            core.require(t[\'candidates_sha256\'] == core.signature(candidates), \'Arbiter candidate set differs\')\n            allowed = set(t[\'requested_candidate_ids\'])\n            candidates = [c for c in candidates if c[\'candidate_id\'] in allowed]\n            core.require(len(candidates) == len(allowed), \'Unknown arbitration candidate\')\n        if candidates:\n            tasks.append((row, query, candidates))\n    return tasks\n\n\ndef validate_request(request, plan, root):\n    core.checked(request)\n    core.require(request[\'plan_signature\'] == plan[\'signature\'] and request[\'schema\'] == \'disagreement-request-v1\',\n                 \'Request belongs to another plan\')\n    core.require(len({t[\'id\'] for t in request[\'tasks\']}) == len(request[\'tasks\']), \'Duplicate request query\')\n    for t in request[\'tasks\']:\n        core.require(len(set(t[\'requested_candidate_ids\'])) == len(t[\'requested_candidate_ids\']), \'Duplicate request candidate\')\n        for source in t[\'source_records\']:\n            path = (Path(root) / source[\'path\']).resolve()\n            core.require(Path(root).resolve() in path.parents and core.file_hash(path) == source[\'sha256\'],\n                         \'Input model scores changed since arbitration request\')\n\n\ndef import_old_qwen(directory, row, query, candidates, plan):\n    """Reuse only proven identical query/passage/policy scores from the prior notebook."""\n    if directory is None:\n        return None\n    directory = Path(directory)\n    manifest = core.checked(core.read(directory / \'manifest.json\'))\n    old = manifest[\'identity\']\n    core.require(old[\'inputs\'] == plan[\'inputs\'] and old[\'generation\'] == plan[\'generation\'] and\n        old[\'model_policy\'] == plan[\'models\'][\'qwen\'] and manifest[\'run_signature\'] == core.signature(old),\n        \'Existing Qwen run differs in input, spans or scoring policy; disable reuse or select matching run\')\n    path = directory / f"query-{row[\'id\']}.json"\n    if not path.is_file() or row[\'id\'] not in manifest[\'completed_query_ids\']:\n        return None\n    core.require(core.file_hash(path) == manifest[\'score_files\'][str(row[\'id\'])], \'Old Qwen score file changed\')\n    record = core.read(path)\n    core.verify_score_record(record,row,query,candidates,manifest[\'run_signature\'],plan[\'models\'][\'qwen\'])\n    return record, dict(path=str(path), sha256=core.file_hash(path), old_run_signature=manifest[\'run_signature\'])\n\n\ndef score_worker(rows, queries, plan, root, role, *, shard_index=0, shard_count=1,\n                 request=None, stop_after_queries=20, local_cache=None, batch_size=8,\n                 padded_tokens=8192, reuse_qwen=None, scorer_factory=None):\n    core.require(stop_after_queries is None or type(stop_after_queries) is int and stop_after_queries>0, \'Invalid pilot limit\')\n    validate_plan(plan, plan[\'inputs\'])\n    if request:\n        validate_request(request,plan,root)\n    identity = worker_identity(plan,role,shard_index,shard_count,request)\n    output = worker_directory(root,identity)\n    output.mkdir(parents=True,exist_ok=True)\n    manifest_path = output/\'manifest.json\'\n    if manifest_path.exists():\n        previous = core.checked(core.read(manifest_path))\n        core.require(previous[\'identity\'] == identity,\'Worker identity differs\')\n    else:\n        core.require(not any(output.iterdir()),\'Nonempty worker directory has no manifest\')\n    tasks = tasks_for(rows,queries,plan,identity,request)\n    policy = plan[\'models\'][role]\n    done, hashes, new, scorer = [], {}, 0, None\n    def save(state):\n        core.write(manifest_path,core.signed(dict(identity=identity,state=state,\n            expected_query_ids=[r[\'id\'] for r,_,_ in tasks],completed_query_ids=done,\n            score_files=hashes,new_queries_last_invocation=new)))\n    save(\'running\')\n    local_cache = Path(local_cache or Path(root)/f\'cache-{role}.sqlite3\')\n    local_cache.parent.mkdir(parents=True,exist_ok=True)\n    with sqlite3.connect(local_cache) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS scores(key TEXT PRIMARY KEY,score REAL NOT NULL)\')\n        try:\n            for row,query,candidates in tasks:\n                path=output/f"query-{row[\'id\']}.json"\n                if path.is_file():\n                    verify_record(core.read(path),row,query,candidates,identity,policy)\n                else:\n                    imported = import_old_qwen(reuse_qwen,row,query,candidates,plan) if role==\'qwen\' else None\n                    if imported:\n                        old, provenance = imported\n                        values, registry, provenance = [s[\'score\'] for s in old[\'scores\']],old[\'model_registry\'],provenance\n                    else:\n                        texts = [core.evidence_text(row,c) for c in candidates]\n                        keys=[core.cache_key(query[\'query\'],text,policy) for text in texts]\n                        unique = dict(zip(keys,texts)); known={}; missing=[]\n                        for key in unique:\n                            hit=db.execute(\'SELECT score FROM scores WHERE key=?\',(key,)).fetchone()\n                            if hit:\n                                core.require(math.isfinite(hit[0]),\'Invalid cached value\')\n                                known[key]=hit[0]\n                            else:\n                                missing.append(key)\n                        if scorer is None:\n                            scorer=(scorer_factory or ModelScorer)(policy,batch_size=batch_size,padded_tokens=padded_tokens)\n                        if missing:\n                            values=scorer.score(query[\'query\'],[unique[k] for k in missing])\n                            core.require(len(values)==len(missing) and all(type(s) in (int,float) and math.isfinite(s) for s in values),\n                                         \'Invalid model output\')\n                            for key,value in zip(missing,values):\n                                db.execute(\'INSERT INTO scores VALUES(?,?)\',(key,float(value))); known[key]=float(value)\n                            db.commit()\n                        values=[known[k] for k in keys]; registry=scorer.registry; provenance=None\n                    record=core.signed(dict(id=row[\'id\'],run_signature=core.signature(identity),\n                        query_sha256=core.digest(query[\'query\'].encode()),candidates_sha256=core.signature(candidates),\n                        model_registry=registry,scores=[dict(candidate_id=c[\'candidate_id\'],score=s) for c,s in zip(candidates,values)],\n                        imported_from=provenance))\n                    verify_record(record,row,query,candidates,identity,policy)\n                    core.write(path,record); new+=1\n                done.append(row[\'id\']); hashes[str(row[\'id\'])]=core.file_hash(path); save(\'running\')\n                print(f"{role} part {shard_index}: query {row[\'id\']} complete {len(done)}/{len(tasks)}, new={new}",flush=True)\n                if stop_after_queries is not None and new>=stop_after_queries and len(done)<len(tasks):\n                    save(\'paused\'); return core.read(manifest_path)\n        except BaseException:\n            save(\'interrupted\'); raise\n    save(\'complete\')\n    return core.read(manifest_path)\n\n\ndef collect_role(rows, queries, plan, root, role, request=None):\n    scope = request[\'signature\'] if request else \'all\'\n    manifests = sorted((Path(root)/\'workers\'/role/scope).glob(\'*/manifest.json\'))\n    records, shard_counts = {},set()\n    by_id={r[\'id\']:(r,q) for r,q in zip(rows,queries)}\n    if request:\n        validate_request(request,plan,root)\n    for path in manifests:\n        m=core.checked(core.read(path)); ident=m[\'identity\']\n        core.require(ident[\'role\']==role and ident==worker_identity(plan,role,ident[\'shard_index\'],ident[\'shard_count\'],request),\n                     \'Worker plan/model/shard differs\')\n        core.require(path.parent.resolve()==worker_directory(root,ident).resolve(),\'Unexpected shard directory\')\n        shard_counts.add(ident[\'shard_count\'])\n        expected={r[\'id\']:cs for r,_,cs in tasks_for(rows,queries,plan,ident,request)}\n        core.require(m[\'expected_query_ids\']==list(expected) and len(set(m[\'completed_query_ids\']))==len(m[\'completed_query_ids\']),\n                     \'Worker query coverage differs\')\n        for qid in m[\'completed_query_ids\']:\n            core.require(qid in expected and qid not in records,\'Unknown or duplicated query across shards\')\n            row,q=by_id[qid]; score_path=path.parent/f\'query-{qid}.json\'\n            core.require(core.file_hash(score_path)==m[\'score_files\'][str(qid)],\'Score file changed\')\n            values=verify_record(core.read(score_path),row,q,expected[qid],ident,plan[\'models\'][role])\n            records[qid]=dict(candidates=expected[qid],scores=values,\n                source=dict(path=score_path.relative_to(root).as_posix(),sha256=core.file_hash(score_path)))\n    core.require(len(shard_counts)<=1,\'Mixed shard counts: use one consistent shard_count per model\')\n    return records\n\n\ndef ranks(candidates,scores):\n    core.require(len(candidates)==len(scores) and len({c[\'candidate_id\'] for c in candidates})==len(candidates)\n                 and all(math.isfinite(s) for s in scores),\'Invalid fusion candidates/scores\')\n    order=sorted(range(len(candidates)),key=lambda i:(-scores[i],candidates[i][\'candidate_id\']))\n    return {candidates[i][\'candidate_id\']:rank for rank,i in enumerate(order,1)}\n\n\ndef fused_scores(candidates, left, right, k=60):\n    core.require(type(k) is int and k>0,\'Positive RRF k required\')\n    a,b=ranks(candidates,left),ranks(candidates,right)\n    return [1/(k+a[c[\'candidate_id\']])+1/(k+b[c[\'candidate_id\']]) for c in candidates]\n\n\ndef disagreement_ids(candidates,left,right, *, top_k=20,rank_gap=30,limit=40):\n    a,b=ranks(candidates,left),ranks(candidates,right)\n    bad=[c[\'candidate_id\'] for c in candidates if\n         (a[c[\'candidate_id\']]<=top_k)!=(b[c[\'candidate_id\']]<=top_k)\n         or abs(a[c[\'candidate_id\']]-b[c[\'candidate_id\']])>=rank_gap]\n    bad.sort(key=lambda key:(-int((a[key]<=top_k)!=(b[key]<=top_k)),-abs(a[key]-b[key]),min(a[key],b[key]),key))\n    return bad[:limit]\n\n\ndef reorder_disputed_slots(candidates,base_scores,requested_ids,arbiter_scores):\n    """Arbiter compares ONLY requested candidates; never grants missing-score bonuses."""\n    by_id={c[\'candidate_id\']:c for c in candidates}\n    core.require(set(requested_ids)<=set(by_id) and len(set(requested_ids))==len(requested_ids)\n                 and len(requested_ids)==len(arbiter_scores) and all(math.isfinite(s) for s in arbiter_scores),\n                 \'Invalid arbitration score coverage\')\n    order=sorted(by_id,key=lambda key:(-base_scores[key],key))\n    slots=[i for i,key in enumerate(order) if key in set(requested_ids)]\n    judged=sorted(zip(requested_ids,arbiter_scores),key=lambda x:(-x[1],x[0]))\n    for index,(key,_) in zip(slots,judged):\n        order[index]=key\n    positions={key:i for i,key in enumerate(order)}\n    return [float(len(order)-positions[c[\'candidate_id\']]) for c in candidates]\n\n\ndef merge_and_export(rows,queries,plan,root,destination,baseline_raw, *, request=None,\n                     top_k=20,rank_gap=30,request_limit=40):\n    validate_plan(plan,plan[\'inputs\'])\n    core.require(core.digest(baseline_raw)==plan[\'inputs\'][\'submission_json_sha256\'],\'Baseline differs\')\n    core.require(type(top_k) is int and top_k>0 and type(rank_gap) is int and rank_gap>0 and\n                 type(request_limit) is int and request_limit>0,\'Invalid disagreement budgets\')\n    qwen=collect_role(rows,queries,plan,root,\'qwen\'); bge=collect_role(rows,queries,plan,root,\'bge\')\n    common={r[\'id\'] for r in rows}&set(qwen)&set(bge)\n    core.require(common,\'No shared completed queries: run both qwen and bge workers first\')\n    full=len(common)==len(rows)\n    arbitration=collect_role(rows,queries,plan,root,\'arbiter\',request) if request else {}\n    request_tasks={t[\'id\']:t for t in request[\'tasks\']} if request else {}\n    arbiter_full=request is not None and full and set(request_tasks)<=set(arbitration) and set(request[\'query_ids\'])==common\n    destination=Path(destination)\n    core.require(not destination.exists(),\'Choose a new merge destination\')\n    destination.mkdir(parents=True)\n    variants=[dict(name=\'Qwen_80_cap2\',kind=\'qwen\',limits=dict(max_chunks=80,per_doc=2,overlap_iou=.8)),\n              dict(name=\'BGE_80_cap2\',kind=\'bge\',limits=dict(max_chunks=80,per_doc=2,overlap_iou=.8)),\n              dict(name=\'RRF_120_cap6\',kind=\'rrf\',limits=dict(max_chunks=120,per_doc=6,overlap_iou=.8)),\n              dict(name=\'RRF_80_cap2\',kind=\'rrf\',limits=dict(max_chunks=80,per_doc=2,overlap_iou=.8))]\n    if arbiter_full:\n        variants += [dict(name=\'Arbitrated_120_cap6\',kind=\'arbiter\',limits=dict(max_chunks=120,per_doc=6,overlap_iou=.8))]\n    predictions={v[\'name\']:[] for v in variants}; stats={v[\'name\']:[] for v in variants}\n    tasks=[]; review_count=0\n    with (destination/\'DISAGREEMENT_REVIEW.jsonl\').open(\'w\',encoding=\'utf-8\') as review, \\\n         (destination/\'EVIDENCE_PROVENANCE.jsonl\').open(\'w\',encoding=\'utf-8\') as provenance:\n        for row,query in zip(rows,queries):\n            qid=row[\'id\']\n            if qid not in common:\n                continue\n            candidates=qwen[qid][\'candidates\']\n            core.require(candidates==bge[qid][\'candidates\'],\'Models scored different candidate pools\')\n            a,b=qwen[qid][\'scores\'],bge[qid][\'scores\']\n            rr=fused_scores(candidates,a,b)\n            requested=disagreement_ids(candidates,a,b,top_k=top_k,rank_gap=rank_gap,limit=request_limit)\n            if requested:\n                tasks.append(dict(id=qid,candidates_sha256=core.signature(candidates),requested_candidate_ids=requested,\n                                  source_records=[qwen[qid][\'source\'],bge[qid][\'source\']]))\n                ra,rb=ranks(candidates,a),ranks(candidates,b)\n                by_id={c[\'candidate_id\']:c for c in candidates}\n                review.write(core.canonical(dict(id=qid,query=query[\'query\'],model_judgments_not_gold=True,\n                    disputed_count=len(requested),examples=[dict(candidate=by_id[key],\n                    text=core.evidence_text(row,by_id[key]),qwen_rank=ra[key],bge_rank=rb[key]) for key in requested[:5]]))+\'\\n\')\n                review_count+=1\n            third=None\n            if arbiter_full and qid in arbitration:\n                task=request_tasks[qid]\n                core.require(task[\'candidates_sha256\']==core.signature(candidates),\'Arbiter full-pool hash differs\')\n                subset=arbitration[qid]\n                ids=[c[\'candidate_id\'] for c in subset[\'candidates\']]\n                third=reorder_disputed_slots(candidates,dict(zip([c[\'candidate_id\'] for c in candidates],rr)),ids,subset[\'scores\'])\n            for v in variants:\n                values={\'qwen\':a,\'bge\':b,\'rrf\':rr,\'arbiter\':third if third is not None else rr}[v[\'kind\']]\n                pred,selected,decisions=core.select_evidence(row,candidates,values,v[\'limits\'])\n                core.require(pred[\'relevant_docs\']==row[\'relevant_docs\'],\'Document list changed\')\n                predictions[v[\'name\']].append(pred)\n                before={(c[\'doc_id\'],c[\'chunk_text\']) for c in row[\'relevant_chunks\']}\n                after={(c[\'doc_id\'],c[\'chunk_text\']) for c in pred[\'relevant_chunks\']}\n                stats[v[\'name\']].append(dict(id=qid,chunks=len(selected),changed=before!=after,\n                    subspans=sum(c[\'kind\']==\'subspan\' for c in selected)))\n                provenance.write(core.canonical(dict(variant=v[\'name\'],id=qid,selected=selected,decisions=decisions))+\'\\n\')\n    new_request=core.signed(dict(schema=\'disagreement-request-v1\',plan_signature=plan[\'signature\'],\n        query_ids=[r[\'id\'] for r in rows if r[\'id\'] in common],scope=\'full\' if full else \'pilot\',\n        rules=dict(top_k=top_k,rank_gap=rank_gap,max_candidates_per_query=request_limit),tasks=tasks))\n    requests_dir=Path(root)/\'requests\'\n    requests_dir.mkdir(parents=True,exist_ok=True)\n    request_path=requests_dir/f"request-{new_request[\'signature\']}.json"\n    if request_path.exists():\n        core.require(core.read(request_path)==new_request,\'Existing request changed\')\n    else:\n        core.write(request_path,new_request)\n    exports=[]\n    if full:\n        with zipfile.ZipFile(destination/\'S0_baseline.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n            z.writestr(\'submission.json\',baseline_raw)\n        for v in variants:\n            payload=core.canonical(predictions[v[\'name\']]).encode()\n            path=destination/(v[\'name\']+\'.zip\')\n            with zipfile.ZipFile(path,\'w\',zipfile.ZIP_DEFLATED) as z:\n                z.writestr(\'submission.json\',payload)\n            with zipfile.ZipFile(path) as z:\n                core.require(z.namelist()==[\'submission.json\'] and z.read(\'submission.json\')==payload,\'Output ZIP differs\')\n            exports.append(dict(variant=v,zip=path.name,zip_sha256=core.file_hash(path),\n                json_sha256=core.digest(payload),queries=len(rows),\n                changed_queries=sum(s[\'changed\'] for s in stats[v[\'name\']]),\n                chunks=sum(s[\'chunks\'] for s in stats[v[\'name\']]),subspans=sum(s[\'subspans\'] for s in stats[v[\'name\']])))\n    report=dict(state=\'full_export\' if full else \'pilot_diagnostics_only\',plan_signature=plan[\'signature\'],\n        total_queries=len(rows),qwen_complete_queries=len(qwen),bge_complete_queries=len(bge),\n        shared_queries=len(common),queries_with_disagreements=review_count,\n        requested_pairs=sum(len(t[\'requested_candidate_ids\']) for t in tasks),request_path=str(request_path),\n        request_signature=new_request[\'signature\'],arbiter_complete=arbiter_full,\n        applied_arbitration_request_signature=request[\'signature\'] if arbiter_full else None,\n        original_corpus_verified=False,relevance_measured=False,official_lcs_evaluated=False,\n        model_judgments_are_ground_truth=False,source_scope=\'exact slices of input submission only\',\n        source_inputs=plan[\'inputs\'],baseline_json_sha256=core.digest(baseline_raw),exports=exports,\n        pilot_statistics=None if full else stats)\n    core.write(destination/\'REPORT.json\',report)\n    return report\n', 'stage9_dense_recovery.py': '"""Recover Stage-8 source bundles and validate dense candidates for frozen S09."""\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport sqlite3\nimport zipfile\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'))\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef require(ok, message):\n    if not ok:\n        raise ValueError(message)\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(canonical(value), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef signed(value):\n    return {**value, \'signature\': digest(canonical(value).encode())}\n\n\ndef checked(value):\n    require(value.get(\'signature\') == signed({k: v for k, v in value.items() if k != \'signature\'})[\'signature\'],\n            \'Signed record checksum differs\')\n    return value\n\n\ndef inspect_bundles(directory, expected):\n    archives = sorted(Path(directory).glob(\'worker_*.zip\'))\n    require(archives, f\'No Stage-8 input bundles in {directory}\')\n    plan, members, metadata = None, {}, {}\n    for archive in archives:\n        with zipfile.ZipFile(archive) as z:\n            names = z.namelist()\n            require(len(names) == len(set(names)), f\'Duplicate ZIP members: {archive}\')\n            current = checked(json.loads(z.read(\'plan.json\')))\n            require(current[\'source\'] == expected, f\'Stage-8 source differs: {archive}\')\n            require(plan is None or current == plan, \'Mixed Stage-8 plans\')\n            plan = current\n            for name, key in [(\'manifest.json\', \'manifest_sha256\'), (\'VALIDATION.json\', \'validation_sha256\')]:\n                data = z.read(\'chunks/\' + name)\n                require(digest(data) == expected[key], f\'Source metadata differs: {archive}/{name}\')\n                metadata[name] = data\n            for part in plan[\'parts\']:\n                index = part[\'index\']\n                prefix = f\'chunks/batch-{index:06d}/\'\n                if prefix + \'documents.jsonl.gz\' not in names:\n                    continue\n                require(prefix + \'batch.json\' in names, f\'Missing batch metadata: {prefix}\')\n                require(index not in members, f\'Duplicate batch across bundles: {index}\')\n                require(digest(z.read(prefix + \'batch.json\')) == part[\'batch_json_sha256\'], \'Batch metadata differs\')\n                members[index] = (archive, prefix, z.getinfo(prefix + \'documents.jsonl.gz\').file_size)\n        print(f\'Inspected {archive.name}; total unique batches: {len(members)}\', flush=True)\n    require(set(members) == set(range(expected[\'batches\'])), \'Missing or unexpected source batches\')\n    require(len(plan[\'parts\']) == expected[\'batches\'], \'Plan batch count differs\')\n    require(sum(p[\'rows\'] for p in plan[\'parts\']) == expected[\'chunks\'] and\n            sum(p[\'documents\'] for p in plan[\'parts\']) == expected[\'documents\'], \'Plan totals differ\')\n    return plan, members, metadata\n\n\ndef restore_chunks(directory, destination, expected):\n    plan, members, metadata = inspect_bundles(directory, expected)\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    for name, data in metadata.items():\n        if (destination / name).exists():\n            require((destination / name).read_bytes() == data, \'Destination contains another source; use a new folder\')\n    missing_bytes = sum(size for index, (_, _, size) in members.items()\n                        if not (destination / f\'batch-{index:06d}/documents.jsonl.gz\').is_file())\n    require(shutil.disk_usage(destination).free > missing_bytes + 8 * 1024**3,\n            \'Insufficient local disk for source plus 8 GiB reserve\')\n    # Open each ZIP once. All output names come from integer indices, never arbitrary ZIP paths.\n    done = 0\n    for archive in sorted({v[0] for v in members.values()}):\n        with zipfile.ZipFile(archive) as z:\n            for index, (owner, prefix, _) in sorted(members.items()):\n                if owner != archive:\n                    continue\n                part = plan[\'parts\'][index]\n                require(part[\'index\'] == index, \'Plan parts are not indexed in order\')\n                target = destination / f\'batch-{index:06d}\'\n                target.mkdir(exist_ok=True)\n                for name, expected_hash in [(\'batch.json\', part[\'batch_json_sha256\']),\n                                            (\'documents.jsonl.gz\', part[\'source_sha256\'])]:\n                    path = target / name\n                    if path.is_file() and file_hash(path) == expected_hash:\n                        continue\n                    temporary = path.with_suffix(path.suffix + \'.partial\')\n                    with z.open(prefix + name) as src, temporary.open(\'wb\') as dst:\n                        shutil.copyfileobj(src, dst, length=4 * 1024**2)\n                    require(file_hash(temporary) == expected_hash, f\'Corrupted payload: {prefix}{name}\')\n                    temporary.replace(path)\n                done += 1\n                if done == 1 or done % 25 == 0 or done == len(members):\n                    print(f\'Restored and verified {done}/{len(members)} batches\', flush=True)\n    require({p.name for p in destination.glob(\'batch-*\')} == {f\'batch-{i:06d}\' for i in members},\n            \'Unexpected destination batches\')\n    for name, data in metadata.items():\n        (destination / name).write_bytes(data)\n    return plan\n\n\nFIELDS = (\'chunk_id\', \'doc_id\', \'derived_version_id\', \'source_batch\', \'source_row\',\n          \'chunk_order\', \'retrieval_text_sha256\')\n\n\ndef adapt_candidates(results, chunks, plan, expected, query_path, queries, output):\n    """Validate every candidate against original text before issuing an adapter report."""\n    results, chunks, output = Path(results), Path(chunks), Path(output)\n    manifest = read(results / \'manifest.json\')\n    ident = manifest[\'identity\']\n    require(manifest[\'signature\'] == digest(canonical(ident).encode()) and manifest[\'state\'] == \'complete\',\n            \'Dense result manifest identity/state differs\')\n    require(ident[\'operation\'] in (\'dense-results\', \'dense-v2-cached-candidates\'), \'Unsupported candidate producer\')\n    require(ident[\'plan_signature\'] == plan[\'signature\'] and plan[\'source\'] == expected, \'Candidate plan/source differs\')\n    require(ident[\'query_sha256\'] == file_hash(query_path), \'Candidate query snapshot differs\')\n    require(manifest.get(\'corpus_complete\') is True, \'Candidate producer reports incomplete corpus\')\n    input_hashes = {str(q[\'id\']): file_hash(results / \'candidates\' / f"query-{q[\'id\']}.json") for q in queries}\n    identity = dict(operation=\'dense-candidate-adapter-v1\', source=expected,\n                    input_manifest_sha256=file_hash(results / \'manifest.json\'), query_sha256=file_hash(query_path),\n                    input_query_hashes=input_hashes, adapter_sha256=file_hash(__file__))\n    signature = digest(canonical(identity).encode())\n    output.mkdir(parents=True, exist_ok=True)\n    if (output / \'manifest.json\').exists():\n        require(read(output / \'manifest.json\')[\'identity\'] == identity, \'Adapter inputs changed; use a new output folder\')\n    write(output / \'manifest.json\', dict(identity=identity, signature=signature, state=\'building\'))\n    # This index is disposable; rebuilt on every validation attempt to avoid stale verification flags.\n    with sqlite3.connect(output / \'verification.sqlite3\') as db:\n        db.execute(\'DROP TABLE IF EXISTS wanted\')\n        db.execute(\'CREATE TABLE wanted(id TEXT PRIMARY KEY,batch INTEGER,payload TEXT,verified INTEGER DEFAULT 0)\')\n        total = 0\n        for number, q in enumerate(queries, 1):\n            path = results / \'candidates\' / f"query-{q[\'id\']}.json"\n            raw = path.read_bytes()\n            require(digest(raw) == input_hashes[str(q[\'id\'])], \'Candidate file changed while reading\')\n            record = checked(json.loads(raw))\n            require(record[\'id\'] == q[\'id\'] and record[\'query\'] == q[\'query\'] and\n                    record[\'run_signature\'] == manifest[\'signature\'], \'Candidate query/run binding differs\')\n            seen, previous = set(), math.inf\n            for item in record[\'candidates\']:\n                require(all(k in item for k in FIELDS), \'Missing candidate mapping\')\n                require(isinstance(item[\'chunk_id\'], str) and item[\'chunk_id\'] and type(item[\'doc_id\']) is int,\n                        \'Invalid candidate ID\')\n                require(all(type(item[k]) is int and item[k] >= 0 for k in (\'source_batch\', \'source_row\', \'chunk_order\')),\n                        \'Invalid source coordinates\')\n                require(item[\'source_batch\'] < expected[\'batches\'], \'Unknown batch\')\n                score = item[\'score\']\n                require(type(score) in (int, float) and math.isfinite(score) and score <= previous,\n                        \'Invalid or unsorted candidate scores\')\n                require(item[\'chunk_id\'] not in seen, \'Duplicate candidate in query\')\n                seen.add(item[\'chunk_id\']); previous = score\n                payload = canonical({k: item[k] for k in FIELDS})\n                old = db.execute(\'SELECT payload FROM wanted WHERE id=?\', (item[\'chunk_id\'],)).fetchone()\n                require(old is None or old[0] == payload, \'Conflicting candidate mapping\')\n                db.execute(\'INSERT OR IGNORE INTO wanted(id,batch,payload) VALUES(?,?,?)\',\n                           (item[\'chunk_id\'], item[\'source_batch\'], payload))\n            total += len(record[\'candidates\'])\n            write(output / f"query-{q[\'id\']}.json", signed(dict(id=q[\'id\'], query=q[\'query\'],\n                  run_signature=signature, candidates=record[\'candidates\'])))\n            if number % 100 == 0:\n                print(f\'Checked candidate records: {number}/{len(queries)}\', flush=True)\n        db.commit()\n        db.execute(\'CREATE INDEX wanted_batch ON wanted(batch)\')\n        for part in plan[\'parts\']:\n            index = part[\'index\']\n            wanted = dict(db.execute(\'SELECT id,payload FROM wanted WHERE batch=?\', (index,)))\n            if not wanted:\n                continue\n            path = chunks / f\'batch-{index:06d}/documents.jsonl.gz\'\n            require(file_hash(path) == part[\'source_sha256\'], \'Restored source payload changed\')\n            matched = set()\n            with gzip.open(path, \'rt\', encoding=\'utf-8\') as stream:\n                for row_number, line in enumerate(stream):\n                    document = json.loads(line)\n                    for order, chunk in enumerate(document[\'chunks\']):\n                        key = chunk[\'chunk_id\']\n                        if key not in wanted:\n                            continue\n                        require(key not in matched, \'Duplicate source chunk ID\')\n                        actual = {**chunk, \'source_batch\': index, \'source_row\': row_number, \'chunk_order\': order,\n                                  \'retrieval_text_sha256\': digest(chunk[\'retrieval_text\'].encode())}\n                        require(canonical({k: actual[k] for k in FIELDS}) == wanted[key], \'Candidate does not match source text\')\n                        require(chunk[\'doc_id\'] == document[\'doc_id\'] and\n                                chunk[\'derived_version_id\'] == document[\'derived_version_id\'] and\n                                digest(chunk[\'chunk_text\'].encode()) == chunk[\'text_sha256\'], \'Source evidence differs\')\n                        matched.add(key)\n            require(matched == set(wanted), f\'Missing evidence in batch {index}\')\n            db.execute(\'UPDATE wanted SET verified=1 WHERE batch=?\', (index,))\n            db.commit()\n            if index % 25 == 0:\n                print(f\'Verified source mappings through batch {index}/{expected["batches"]}\', flush=True)\n        require(db.execute(\'SELECT COUNT(*) FROM wanted WHERE verified=0\').fetchone()[0] == 0, \'Unverified candidates\')\n    write(output / \'manifest.json\', dict(identity=identity, signature=signature, state=\'complete\',\n                                        queries=len(queries), candidates=total))\n    report = dict(validation=\'passed\', validation_scope=\'candidate integrity and exact source mapping; not relevance\',\n                  manifest_sha256=file_hash(output / \'manifest.json\'), query_sha256=file_hash(query_path),\n                  queries=len(queries), candidates=total, relevance_measured=False)\n    write(output / \'VALIDATION.json\', report)\n    return report\n', 'corpus_multiretrieval.py': '"""Source-bound BM25 + three dense retrievers, two rerankers and submission export.\n\nDrive checkpoints contain source/model/code identities. No submission ZIP is an input.\n"""\nfrom collections import Counter, defaultdict\nimport gzip\nimport io\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport shutil\nimport sqlite3\nimport zipfile\n\ntry:\n    from . import qwen_submission_trial as core\n    from . import multimodel_evidence as multi\n    from . import stage9_dense_recovery as recovery\nexcept ImportError:\n    import qwen_submission_trial as core\n    import multimodel_evidence as multi\n    import stage9_dense_recovery as recovery\n\nSOURCE = dict(signature=\'5ce3d7c02e6a0a8275e203a4cff9c12fb371fc8cf0fd9515574250acfff3751c\',\n    manifest_sha256=\'d88fd1c046a93950c7e97ef5c8786d933a677ef1f54e7880f5c34fa6c4ff40f1\',\n    validation_sha256=\'5bdc88900f47989a4edb653ae1f6b90ba90a79760f620edb35ac09a48d20d993\',\n    documents=855745, chunks=5512624, batches=1712)\nLINKS_SHA = \'87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7\'\nBRANCHES = (\'bm25\', \'bge\', \'qwen\', \'e5\')\n\n\ndef embedding_policies():\n    return dict(\n        bge=dict(model_id=\'BAAI/bge-m3\', revision=\'5617a9f61b028005a4858fdac845db406aefb181\',\n                 date=\'2024-07-03T14:50:10Z\', pooling=\'cls\', max_tokens=1024, safetensors=False),\n        qwen=dict(model_id=\'Qwen/Qwen3-Embedding-0.6B\', revision=\'97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3\',\n                  date=\'2026-04-20T02:45:58Z\', pooling=\'last\', max_tokens=2048,\n                  padding=\'left\', attention=\'eager\', dtype=\'bfloat16_if_supported_else_float16\'),\n        e5=dict(model_id=\'intfloat/multilingual-e5-large\', revision=\'3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3\',\n                date=\'2026-04-02T02:16:59Z\', pooling=\'mean\', max_tokens=512))\n\n\ndef make_plan(query_path, queries, query_limit=20, batch_limit=4, top_k=500, rerank_k=400):\n    import pyarrow.parquet as pq\n    core.require(query_limit is None or type(query_limit) is int and query_limit > 0, \'Invalid query limit\')\n    core.require(batch_limit is None or type(batch_limit) is int and 0 < batch_limit <= SOURCE[\'batches\'], \'Invalid batch limit\')\n    core.require(type(top_k) is int and type(rerank_k) is int and 0 < rerank_k <= top_k, \'Invalid pool sizes\')\n    core.require(core.file_hash(query_path) == core.EXPECTED_QUERY_SHA256 and len(queries) == 1200,\n                 \'Use the original official 1200-query snapshot\')\n    core.require(queries == pq.read_table(query_path, columns=[\'id\', \'query\']).to_pylist(), \'Query text/order differs from file\')\n    selected = queries if query_limit is None else queries[:query_limit]\n    return core.signed(dict(operation=\'corpus-multiretrieval-v1\', source=SOURCE,\n        query_sha256=core.file_hash(query_path), queries=selected,\n        batch_indices=list(range(SOURCE[\'batches\'] if batch_limit is None else batch_limit)),\n        full_scope=len(selected) == 1200 and batch_limit is None,\n        top_k=top_k, rerank_k=rerank_k, embeddings=embedding_policies(), rerankers=multi.policies(),\n        dense_scoring=\'normalized-float32-inner-product\', long_text=\'overlap64-token-windows-weighted-mean-v1\',\n        bm25=\'sqlite-fts5-global-bm25-unicode61-plus-cjk-bigrams-v1\', bm25_query_terms=128,\n        code={name: core.file_hash(path) for name, path in dict(\n            pipeline=__file__, core=core.__file__, multi=multi.__file__, recovery=recovery.__file__).items()}))\n\n\ndef check_plan(plan):\n    core.checked(plan)\n    core.require(plan[\'source\'] == SOURCE and plan[\'embeddings\'] == embedding_policies()\n        and plan[\'rerankers\'] == multi.policies(), \'Source/model policy changed\')\n    core.require(plan[\'batch_indices\'] == list(range(len(plan[\'batch_indices\']))) and\n        0 < len(plan[\'batch_indices\']) <= SOURCE[\'batches\'], \'Invalid source scope\')\n    core.require(plan[\'full_scope\'] == (len(plan[\'queries\']) == 1200 and\n        len(plan[\'batch_indices\']) == SOURCE[\'batches\']), \'Full/pilot scope differs\')\n    for name, path in dict(pipeline=__file__, core=core.__file__, multi=multi.__file__, recovery=recovery.__file__).items():\n        core.require(plan[\'code\'][name] == core.file_hash(path), \'Support code changed; use a new run\')\n\n\ndef setup_source(directory, plan):\n    check_plan(plan)\n    source_plan, members, _ = recovery.inspect_bundles(directory, plan[\'source\'])\n    return source_plan, members\n\n\ndef source_batch(source_plan, members, index):\n    part = source_plan[\'parts\'][index]\n    archive, prefix, _ = members[index]\n    with zipfile.ZipFile(archive) as z:\n        raw = z.read(prefix + \'documents.jsonl.gz\')\n    core.require(core.digest(raw) == part[\'source_sha256\'], f\'Source batch changed: {index}\')\n    rows, documents = [], 0\n    with gzip.open(io.BytesIO(raw), \'rt\', encoding=\'utf-8\') as stream:\n        for row_index, line in enumerate(stream):\n            document = json.loads(line)\n            documents += 1\n            for order, chunk in enumerate(document[\'chunks\']):\n                core.require(chunk[\'doc_id\'] == document[\'doc_id\'] and\n                    chunk[\'derived_version_id\'] == document[\'derived_version_id\'] and\n                    core.digest(chunk[\'chunk_text\'].encode()) == chunk[\'text_sha256\'], \'Source evidence checksum differs\')\n                core.require(type(chunk[\'doc_id\']) is int and bool(chunk[\'retrieval_text\'].strip())\n                    and bool(chunk[\'chunk_text\'].strip()), \'Invalid source text/ID\')\n                rows.append({**chunk, \'source_batch\': index, \'source_row\': row_index, \'source_chunk_order\': order})\n    core.require(len(rows) == part[\'rows\'] and documents == part[\'documents\'], \'Source counts differ\')\n    return rows\n\n\ndef assigned(plan, shard, shards):\n    core.require(type(shards) is int and 1 <= shards <= 32 and type(shard) is int and 0 <= shard < shards, \'Invalid shard\')\n    return [i for i in plan[\'batch_indices\'] if i % shards == shard]\n\n\ndef branch_root(run, role, shard, shards):\n    return Path(run) / \'retrieval\' / role / f\'part-{shard}-of-{shards}\'\n\n\ndef trim(hits, k):\n    return dict(sorted(hits.items(), key=lambda p: (-p[1], p[0]))[:k])\n\n\ndef lexical(text):\n    # Preserve Vietnamese words/numbers. Add Chinese unigrams and adjacent bigrams.\n    return \' \'.join(re.findall(r\'[^\\W_]+\', re.sub(r\'[\\u3400-\\u9fff]+\', \' \', text.lower()), flags=re.UNICODE)\n        + [token for run in re.findall(r\'[\\u3400-\\u9fff]+\', text)\n           for token in list(run) + [run[i:i+2] for i in range(len(run)-1)]])\n\n\ndef validate_embedding_batch(batch, vocabulary_size, max_tokens):\n    """Check IDs and gather positions on CPU before any CUDA indexing."""\n    import torch\n    ids, mask = batch[\'input_ids\'], batch[\'attention_mask\']\n    core.require(ids.device.type == mask.device.type == \'cpu\', \'Validate token batch before CUDA transfer\')\n    core.require(ids.ndim == 2 and ids.shape == mask.shape and 0 < ids.shape[1] <= max_tokens,\n                 \'Invalid embedding token/mask shape or context length\')\n    core.require(ids.dtype in (torch.int64, torch.int32), \'Token IDs must be integers\')\n    core.require(int(ids.min()) >= 0 and int(ids.max()) < vocabulary_size,\n        f\'Token IDs outside model vocabulary: min={int(ids.min())}, max={int(ids.max())}, vocabulary={vocabulary_size}\')\n    core.require(bool(((mask == 0) | (mask == 1)).all()) and bool((mask.sum(1) > 0).all()),\n                 \'Attention mask must be binary with nonempty sequences\')\n    return [int(row.nonzero()[-1]) for row in mask]\n\n\ndef pool_embeddings(hidden, mask, pooling, last_positions):\n    import torch\n    core.require(hidden.ndim == 3 and hidden.shape[:2] == mask.shape, \'Hidden state and mask shape differ\')\n    if pooling == \'cls\':\n        return hidden[:, 0]\n    if pooling == \'mean\':\n        return (hidden.float()*mask[..., None]).sum(1)/mask.sum(1)[:, None]\n    core.require(pooling == \'last\' and len(last_positions) == hidden.shape[0] and\n        all(0 <= i < hidden.shape[1] for i in last_positions), \'Invalid last-token pooling positions\')\n    positions = torch.tensor(last_positions, device=hidden.device, dtype=torch.long)\n    return hidden.gather(1, positions[:, None, None].expand(-1, 1, hidden.shape[2])).squeeze(1)\n\n\nclass Encoder:\n    """Native model pooling; all tokens covered by windows, never silent truncation."""\n    def __init__(self, policy, batch_size=16):\n        import numpy as np\n        import torch\n        from transformers import AutoModel, AutoTokenizer\n        core.require(torch.cuda.is_available(), \'Embedding worker requires CUDA\')\n        self.np, self.torch, self.policy, self.batch_size = np, torch, policy, batch_size\n        self.tokenizer = AutoTokenizer.from_pretrained(policy[\'model_id\'], revision=policy[\'revision\'],\n            trust_remote_code=False, padding_side=policy.get(\'padding\', \'right\'))\n        dtype = torch.bfloat16 if policy.get(\'dtype\', \'\').startswith(\'bfloat16\') and torch.cuda.is_bf16_supported() else torch.float16\n        self.model = AutoModel.from_pretrained(policy[\'model_id\'], revision=policy[\'revision\'],\n            trust_remote_code=False, dtype=dtype, use_safetensors=policy.get(\'safetensors\', True),\n            attn_implementation=policy.get(\'attention\', \'sdpa\')).cuda().eval()\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        core.require(self.model.config._commit_hash == policy[\'revision\'], \'Embedding revision differs\')\n        core.require(sum(p.numel() for p in self.model.parameters()) <= 15_000_000_000, \'Model exceeds parameter limit\')\n        self.vocabulary_size = self.model.get_input_embeddings().num_embeddings\n        print(\'Embedding runtime:\', dict(model=policy[\'model_id\'], gpu=torch.cuda.get_device_name(0),\n            torch=torch.__version__, cuda=torch.version.cuda, dtype=str(dtype),\n            attention=policy.get(\'attention\', \'sdpa\'), padding=self.tokenizer.padding_side,\n            vocabulary=self.vocabulary_size), flush=True)\n\n    def encode(self, texts, query=False):\n        import numpy as np\n        t, tok, policy = self.torch, self.tokenizer, self.policy\n        features, owners, weights = [], [], []\n        for owner, text in enumerate(texts):\n            prefix = \'\'\n            if policy[\'pooling\'] == \'mean\':\n                prefix = \'query: \' if query else \'passage: \'\n            elif policy[\'pooling\'] == \'last\' and query:\n                prefix = \'Instruct: Given a web search query, retrieve relevant passages that answer the query\\nQuery: \'\n            header = tok.encode(prefix, add_special_tokens=False)\n            body = tok.encode(text, add_special_tokens=False)\n            capacity = policy[\'max_tokens\'] - len(header) - tok.num_special_tokens_to_add(pair=False)\n            core.require(capacity > 64 and body, \'Invalid token budget/empty text\')\n            start = 0\n            while start < len(body):\n                end = min(start+capacity, len(body))\n                ids = tok.build_inputs_with_special_tokens(header+body[start:end])\n                features.append(dict(input_ids=ids, attention_mask=[1]*len(ids)))\n                owners.append(owner)\n                weights.append(end-start if start == 0 else end-start-64)\n                if end == len(body):\n                    break\n                start = end-64\n        output, pos = None, 0\n        while pos < len(features):\n            size = min(self.batch_size, len(features)-pos)\n            stage = \'CPU token validation\'\n            try:\n                batch = tok.pad(features[pos:pos+size], padding=True, return_tensors=\'pt\')\n                last_positions = validate_embedding_batch(batch, self.vocabulary_size, policy[\'max_tokens\'])\n                stage = \'CUDA transfer / model forward\'\n                batch = batch.to(\'cuda\')\n                with t.inference_mode():\n                    kwargs = dict(use_cache=False) if policy[\'pooling\'] == \'last\' else {}\n                    hidden = self.model(**batch, **kwargs).last_hidden_state\n                    t.cuda.synchronize()\n                    stage = \'token pooling\'\n                    mask = batch[\'attention_mask\']\n                    vectors = pool_embeddings(hidden, mask, policy[\'pooling\'], last_positions).float().cpu().numpy()\n                    stage = \'CPU vector normalization\'\n                    core.require(np.isfinite(vectors).all(), \'Model returned nonfinite embeddings\')\n                    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)\n                    core.require(bool((lengths > 0).all()), \'Model returned zero embeddings\')\n                    vectors = vectors/lengths\n                if output is None:\n                    output = np.zeros((len(texts), vectors.shape[1]), dtype=np.float32)\n                for j, vector in enumerate(vectors):\n                    output[owners[pos+j]] += vector*weights[pos+j]\n                pos += size\n            except t.cuda.OutOfMemoryError:\n                if size == 1:\n                    raise\n                self.batch_size = max(1, size//2)\n                batch, hidden, vectors = None, None, None\n                t.cuda.empty_cache()\n            except RuntimeError as exc:\n                if \'device-side assert\' not in str(exc) and \'CUDA error\' not in str(exc):\n                    raise\n                # Do not call CUDA again: a device assertion poisons this context.\n                raise RuntimeError(f\'Embedding failed at {stage}; model={policy["model_id"]}, \'\n                    f\'query={query}, window_offset={pos}, batch_size={size}. \'\n                    \'Restart the Colab session before retrying. Set DEBUG_CUDA=True before importing torch \'\n                    \'for synchronous diagnostics. Original error: \'+str(exc)) from exc\n        norms = np.linalg.norm(output, axis=1, keepdims=True)\n        core.require(np.all(norms > 0) and np.isfinite(output).all(), \'Invalid embedding vectors\')\n        return output/norms\n\n\ndef dense_top(query_vectors, vectors, k):\n    import numpy as np\n    # Per-source-batch exact cosine search. Bounded CPU RAM; use GPU if available.\n    k = min(k, len(vectors))\n    if not k:\n        return [[] for _ in query_vectors]\n    import torch\n    result = []\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    bank = torch.as_tensor(vectors, dtype=torch.float32, device=device)\n    for start in range(0, len(query_vectors), 32):\n        q = torch.as_tensor(query_vectors[start:start+32], dtype=torch.float32, device=device)\n        scores = (q @ bank.T).cpu().numpy()\n        for row in scores:\n            # Stable source order resolves score ties.\n            indices = np.argsort(-row, kind=\'stable\')[:k]\n            result.append([(int(i), float(row[i])) for i in indices])\n    return result\n\n\ndef retrieve_dense(plan, run, role, source_plan, members, shard=0, shards=1,\n                   stop_after_batches=2, save_embeddings=True, encoder_factory=Encoder):\n    import numpy as np\n    check_plan(plan)\n    core.require(role in plan[\'embeddings\'], \'Unknown dense role\')\n    parts = assigned(plan, shard, shards)\n    core.require(parts, \'Empty shard; reduce SHARD_COUNT\')\n    output = branch_root(run, role, shard, shards)\n    output.mkdir(parents=True, exist_ok=True)\n    identity = dict(plan=plan[\'signature\'], role=role, shard=shard, shards=shards,\n                    save_embeddings=save_embeddings)\n    checkpoint = output/\'checkpoint.json\'\n    saved = core.checked(core.read(checkpoint)) if checkpoint.exists() else None\n    if saved:\n        core.require(saved[\'identity\'] == identity and saved[\'completed_batches\'] == parts[:len(saved[\'completed_batches\'])], \'Retrieval checkpoint identity/coverage differs\')\n    completed = saved[\'completed_batches\'] if saved else []\n    hits = saved[\'hits\'] if saved else [{} for _ in plan[\'queries\']]\n    core.require(len(hits) == len(plan[\'queries\']), \'Query results differ\')\n    encoder = None\n    def encode(texts, query=False):\n        nonlocal encoder\n        if encoder is None:\n            encoder = encoder_factory(plan[\'embeddings\'][role])\n        return encoder.encode(texts, query=query)\n    new = 0\n    qvectors = None\n    for index in parts[len(completed):]:\n        if stop_after_batches is not None and new >= stop_after_batches:\n            break\n        rows = source_batch(source_plan, members, index)\n        if not rows:\n            completed.append(index)\n            continue\n        if qvectors is None:\n            qvectors = encode([q[\'query\'] for q in plan[\'queries\']], query=True)\n        vector_path = output/f\'embeddings-{index:06d}.npy\'\n        receipt_path = output/f\'embeddings-{index:06d}.json\'\n        if vector_path.is_file() and receipt_path.is_file():\n            receipt = core.checked(core.read(receipt_path))\n            core.require(receipt[\'identity\'] == identity and receipt[\'batch\'] == index and\n                receipt[\'sha256\'] == core.file_hash(vector_path), \'Embedding receipt differs\')\n            vectors = np.load(vector_path, allow_pickle=False)\n        else:\n            vectors = encode([row[\'retrieval_text\'] for row in rows])\n            if save_embeddings:\n                temporary = vector_path.with_suffix(\'.npy.tmp\')\n                with temporary.open(\'wb\') as stream:\n                    np.save(stream, vectors, allow_pickle=False)\n                temporary.replace(vector_path)\n                core.write(receipt_path, core.signed(dict(identity=identity, batch=index,\n                    sha256=core.file_hash(vector_path))))\n        core.require(vectors.shape == (len(rows), qvectors.shape[1]) and np.isfinite(vectors).all(), \'Embedding shape/value differs\')\n        for q, result in enumerate(dense_top(qvectors, vectors, plan[\'top_k\'])):\n            for position, score in result:\n                key = rows[position][\'chunk_id\']\n                hits[q][key] = max(score, hits[q].get(key, -float(\'inf\')))\n            hits[q] = trim(hits[q], plan[\'top_k\'])\n        completed.append(index)\n        new += 1\n        # Exact prefix is saved after each batch; no gzip timestamps in identity.\n        core.write(checkpoint, core.signed(dict(identity=identity, completed_batches=completed, hits=hits)))\n        print(f\'{role} shard {shard}/{shards}: {len(completed)}/{len(parts)} batches\', flush=True)\n    result = core.signed(dict(identity=identity, state=\'complete\' if completed == parts else \'paused\',\n        completed_batches=completed, hits=hits))\n    core.write(output/\'result.json\', result)\n    return result\n\n\ndef retrieve_bm25(plan, run, source_plan, members, local_db, stop_after_batches=2):\n    check_plan(plan)\n    # One global FTS index: shard-local BM25 scores are not comparable.\n    output = branch_root(run, \'bm25\', 0, 1)\n    output.mkdir(parents=True, exist_ok=True)\n    destination = output/\'bm25.sqlite3\'\n    local_db = Path(local_db)\n    local_db.parent.mkdir(parents=True, exist_ok=True)\n    if not local_db.exists() and destination.exists():\n        shutil.copy2(destination, local_db)\n    with sqlite3.connect(local_db) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS metadata(key TEXT PRIMARY KEY,value TEXT)\')\n        old = db.execute(\'SELECT value FROM metadata WHERE key=?\', (\'plan\',)).fetchone()\n        core.require(old is None or old[0] == plan[\'signature\'], \'Local BM25 database belongs to another plan\')\n        db.execute(\'INSERT OR IGNORE INTO metadata VALUES(?,?)\', (\'plan\', plan[\'signature\']))\n        db.execute("CREATE VIRTUAL TABLE IF NOT EXISTS passages USING fts5(chunk_id UNINDEXED,text,tokenize=\'unicode61 remove_diacritics 2\')")\n        db.execute(\'CREATE TABLE IF NOT EXISTS batches(id INTEGER PRIMARY KEY)\')\n        completed = [row[0] for row in db.execute(\'SELECT id FROM batches ORDER BY id\')]\n        core.require(completed == plan[\'batch_indices\'][:len(completed)], \'BM25 batch prefix differs\')\n        new = 0\n        for index in plan[\'batch_indices\'][len(completed):]:\n            if stop_after_batches is not None and new >= stop_after_batches:\n                break\n            rows = source_batch(source_plan, members, index)\n            with db:\n                db.executemany(\'INSERT INTO passages(chunk_id,text) VALUES(?,?)\',\n                    [(row[\'chunk_id\'], lexical(row[\'retrieval_text\'])) for row in rows])\n                db.execute(\'INSERT INTO batches VALUES(?)\', (index,))\n            completed.append(index)\n            new += 1\n            print(f\'BM25: {len(completed)}/{len(plan["batch_indices"])} batches\', flush=True)\n        hits = []\n        if completed == plan[\'batch_indices\']:\n            for query in plan[\'queries\']:\n                terms = list(dict.fromkeys(lexical(query[\'query\']).split()))[:plan[\'bm25_query_terms\']]\n                expression = \' OR \'.join(\'"\'+term.replace(\'"\', \'""\')+\'"\' for term in terms)\n                results = db.execute(\'SELECT chunk_id,bm25(passages) AS score FROM passages WHERE passages MATCH ? ORDER BY score,chunk_id LIMIT ?\',\n                    (expression, plan[\'top_k\'])).fetchall() if terms else []\n                hits.append(dict((key, -float(score)) for key, score in results))\n    # Snapshot only at cell boundary. Each local batch transaction is resumable.\n    pending = destination.with_suffix(\'.sqlite3.tmp\')\n    shutil.copy2(local_db, pending)\n    core.require(core.file_hash(pending) == core.file_hash(local_db), \'BM25 backup differs\')\n    pending.replace(destination)\n    result = core.signed(dict(identity=dict(plan=plan[\'signature\'], role=\'bm25\', shard=0, shards=1),\n        state=\'complete\' if completed == plan[\'batch_indices\'] else \'paused\', completed_batches=completed, hits=hits))\n    core.write(output/\'result.json\', result)\n    return result\n\n\ndef collect_branch(plan, run, role):\n    check_plan(plan)\n    files = sorted((Path(run)/\'retrieval\'/role).glob(\'part-*/result.json\'))\n    core.require(files, f\'No retrieval results for {role}\')\n    hits, batches, shard_count = [dict() for _ in plan[\'queries\']], set(), None\n    for path in files:\n        result = core.checked(core.read(path))\n        identity = result[\'identity\']\n        core.require(identity[\'plan\'] == plan[\'signature\'] and identity[\'role\'] == role\n            and result[\'state\'] == \'complete\', f\'Incomplete/wrong retrieval worker: {path}\')\n        core.require(path.parent == branch_root(run, role, identity[\'shard\'], identity[\'shards\']), \'Wrong shard directory\')\n        core.require(shard_count is None or shard_count == identity[\'shards\'], \'Mixed shard counts; use a new plan directory\')\n        shard_count = identity[\'shards\']\n        expected = assigned(plan, identity[\'shard\'], shard_count)\n        core.require(result[\'completed_batches\'] == expected and not batches.intersection(expected), \'Missing/duplicate source coverage\')\n        batches.update(expected)\n        core.require(len(result[\'hits\']) == len(hits), \'Query results missing\')\n        for q, values in enumerate(result[\'hits\']):\n            core.require(all(isinstance(s, (int, float)) and math.isfinite(s) for s in values.values()), \'Invalid retrieval scores\')\n            for key, score in values.items():\n                hits[q][key] = max(score, hits[q].get(key, -float(\'inf\')))\n            hits[q] = trim(hits[q], plan[\'top_k\'])\n    core.require(batches == set(plan[\'batch_indices\']), f\'Missing {role} shards\')\n    return hits\n\n\ndef fuse_candidates(branches, k):\n    scores, provenance = defaultdict(float), defaultdict(dict)\n    for branch, hits in branches.items():\n        for rank, (key, score) in enumerate(sorted(hits.items(), key=lambda p: (-p[1], p[0])), 1):\n            scores[key] += 1/(60+rank)\n            provenance[key][branch] = dict(rank=rank, score=score)\n    return [dict(chunk_id=key, retrieval_rrf=score, branches=provenance[key]) for key, score in\n            sorted(scores.items(), key=lambda p: (-p[1], p[0]))[:k]]\n\n\ndef prepare_pool(plan, run, source_plan, members, local_db):\n    branch_hits = {role: collect_branch(plan, run, role) for role in BRANCHES}\n    pools = [fuse_candidates({role: hits[q] for role, hits in branch_hits.items()}, plan[\'rerank_k\'])\n             for q in range(len(plan[\'queries\']))]\n    diagnostics = []\n    for index, (query, pool) in enumerate(zip(plan[\'queries\'], pools)):\n        sets = {role: set(hits[index]) for role, hits in branch_hits.items()}\n        diagnostics.append(dict(id=query[\'id\'], union_candidates=len(set().union(*sets.values())),\n            selected_for_rerank=len(pool), branches={role: dict(retrieved=len(keys),\n                exclusive_candidates=len(keys-set().union(*(other for name, other in sets.items() if name != role))),\n                represented_in_rerank_pool=sum(role in c[\'branches\'] for c in pool)) for role, keys in sets.items()}))\n    wanted = {item[\'chunk_id\'] for pool in pools for item in pool}\n    local_db = Path(local_db)\n    local_db.parent.mkdir(parents=True, exist_ok=True)\n    with sqlite3.connect(local_db) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS evidence(id TEXT PRIMARY KEY,payload TEXT)\')\n        db.execute(\'DELETE FROM evidence\')\n        for index in plan[\'batch_indices\']:\n            for row in source_batch(source_plan, members, index):\n                if row[\'chunk_id\'] in wanted:\n                    db.execute(\'INSERT INTO evidence VALUES(?,?)\', (row[\'chunk_id\'], core.canonical(row)))\n            if index % 25 == 0:\n                print(f\'Hydrate original evidence: batch {index}\', flush=True)\n        db.commit()\n        core.require(db.execute(\'SELECT COUNT(*) FROM evidence\').fetchone()[0] == len(wanted), \'Candidate missing from source\')\n        output, hashes = Path(run)/\'prepared\', {}\n        output.mkdir(parents=True, exist_ok=True)\n        for query, pool in zip(plan[\'queries\'], pools):\n            candidates = []\n            for item in pool:\n                raw = db.execute(\'SELECT payload FROM evidence WHERE id=?\', (item[\'chunk_id\'],)).fetchone()\n                candidates.append({**json.loads(raw[0]), **item})\n            core.require(candidates, f\'No candidates for query {query["id"]}\')\n            path = output/f\'query-{query["id"]}.json\'\n            core.write(path, core.signed(dict(plan=plan[\'signature\'], **query, candidates=candidates)))\n            hashes[str(query[\'id\'])] = core.file_hash(path)\n    manifest = core.signed(dict(plan=plan[\'signature\'], state=\'complete\', files=hashes,\n        original_evidence_verified=True, full_scope=plan[\'full_scope\'], queries=len(pools)))\n    core.write(output/\'manifest.json\', manifest)\n    core.write(output/\'RETRIEVAL_DIAGNOSTICS.json\', diagnostics)\n    return manifest\n\n\ndef packet(plan, run, query):\n    manifest = core.checked(core.read(Path(run)/\'prepared/manifest.json\'))\n    path = Path(run)/\'prepared\'/f\'query-{query["id"]}.json\'\n    core.require(manifest[\'plan\'] == plan[\'signature\'] and manifest[\'original_evidence_verified\']\n        and manifest[\'files\'][str(query[\'id\'])] == core.file_hash(path), \'Prepared evidence changed\')\n    value = core.checked(core.read(path))\n    core.require(value[\'plan\'] == plan[\'signature\'] and value[\'query\'] == query[\'query\'] and value[\'id\'] == query[\'id\'], \'Query/packet differs\')\n    return value\n\n\ndef rerank(plan, run, role, shard=0, shards=1, stop_after_queries=5, scorer_factory=multi.ModelScorer):\n    check_plan(plan)\n    core.require(role in (\'qwen\', \'bge\') and 0 <= shard < shards <= 32, \'Invalid rerank role/shard\')\n    output = Path(run)/\'rerank\'/role\n    output.mkdir(parents=True, exist_ok=True)\n    scorer, completed, new = None, [], 0\n    for index, query in enumerate(plan[\'queries\']):\n        if index % shards != shard:\n            continue\n        item = packet(plan, run, query)\n        path = output/f\'query-{query["id"]}.json\'\n        if path.exists():\n            score_record(plan, item, core.read(path), role)\n            completed.append(query[\'id\'])\n            continue\n        if stop_after_queries is not None and new >= stop_after_queries:\n            break\n        if scorer is None:\n            scorer = scorer_factory(plan[\'rerankers\'][role])\n        scores = scorer.score(query[\'query\'], [c[\'retrieval_text\'] for c in item[\'candidates\']])\n        record = core.signed(dict(plan=plan[\'signature\'], packet_signature=item[\'signature\'], role=role,\n            model=plan[\'rerankers\'][role], id=query[\'id\'], scores=scores))\n        score_record(plan, item, record, role)\n        core.write(path, record)\n        completed.append(query[\'id\'])\n        new += 1\n        print(f\'Rerank {role}: query {query["id"]}, {len(scores)} candidates\', flush=True)\n    return dict(role=role, shard=shard, shards=shards, completed_ids=completed, new_queries=new)\n\n\ndef score_record(plan, item, value, role):\n    core.checked(value)\n    core.require(value[\'plan\'] == plan[\'signature\'] and value[\'packet_signature\'] == item[\'signature\']\n        and value[\'role\'] == role and value[\'id\'] == item[\'id\'] and value[\'model\'] == plan[\'rerankers\'][role], \'Rerank identity differs\')\n    core.require(len(value[\'scores\']) == len(item[\'candidates\']) and\n        all(type(s) in (int, float) and math.isfinite(s) for s in value[\'scores\']), \'Invalid rerank scores\')\n    return value[\'scores\']\n\n\ndef choose(item, scores, max_docs=120, max_chunks=120, per_doc=6):\n    ranked = sorted(range(len(scores)), key=lambda i: (-scores[i], item[\'candidates\'][i][\'chunk_id\']))\n    docs, selected, seen, counts, versions = [], [], set(), Counter(), {}\n    for i in ranked:\n        c = item[\'candidates\'][i]\n        d = c[\'doc_id\']\n        if d not in docs and len(docs) < max_docs:\n            docs.append(d)\n    for i in ranked:\n        c = item[\'candidates\'][i]\n        d = c[\'doc_id\']\n        key = (d, c[\'text_sha256\'])\n        if d not in docs or len(selected) >= max_chunks or counts[d] >= per_doc or key in seen:\n            continue\n        if d in versions and versions[d] != c[\'derived_version_id\']:\n            continue\n        overlap = False\n        for peer in selected:\n            if (d, c[\'derived_version_id\'], c.get(\'region_id\')) != (peer[\'doc_id\'], peer[\'derived_version_id\'], peer.get(\'region_id\')):\n                continue\n            if all(k in c and k in peer for k in (\'source_start\', \'source_end\')):\n                intersection = max(0, min(c[\'source_end\'], peer[\'source_end\'])-max(c[\'source_start\'], peer[\'source_start\']))\n                union = max(c[\'source_end\'], peer[\'source_end\'])-min(c[\'source_start\'], peer[\'source_start\'])\n                if union > 0 and intersection/union >= .8:\n                    overlap = True\n                    break\n        if overlap:\n            continue\n        selected.append(c)\n        seen.add(key)\n        counts[d] += 1\n        versions[d] = c[\'derived_version_id\']\n    prediction = dict(id=item[\'id\'], relevant_docs=docs, relevant_chunks=[\n        dict(doc_id=c[\'doc_id\'], chunk_text=c[\'chunk_text\'], chunk_order=i) for i, c in enumerate(selected)])\n    return prediction, selected\n\n\ndef merge_export(plan, run, destination, allowed_ids=None):\n    check_plan(plan)\n    destination = Path(destination)\n    core.require(not destination.exists(), \'Use a new output directory\')\n    core.require(not plan[\'full_scope\'] or allowed_ids is not None, \'Full submission requires official document allowlist\')\n    predictions, provenance = [], []\n    for query in plan[\'queries\']:\n        item = packet(plan, run, query)\n        ranks = defaultdict(float)\n        for role in (\'qwen\', \'bge\'):\n            path = Path(run)/\'rerank\'/role/f\'query-{query["id"]}.json\'\n            core.require(path.is_file(), f\'Finish {role} reranking query {query["id"]}\')\n            scores = score_record(plan, item, core.read(path), role)\n            for rank, i in enumerate(sorted(range(len(scores)), key=lambda i: (-scores[i], item[\'candidates\'][i][\'chunk_id\'])), 1):\n                ranks[i] += 1/(60+rank)\n        pred, selected = choose(item, [ranks[i] for i in range(len(item[\'candidates\']))])\n        core.require(pred[\'relevant_chunks\'], \'Empty evidence prediction\')\n        if allowed_ids is not None:\n            core.require(set(pred[\'relevant_docs\']) <= allowed_ids, \'Document ID absent from official links\')\n        predictions.append(pred)\n        provenance.append(dict(id=query[\'id\'], selected=[{k: c.get(k) for k in (\n            \'chunk_id\', \'doc_id\', \'derived_version_id\', \'source_batch\', \'source_row\', \'source_chunk_order\',\n            \'region_id\', \'source_start\', \'source_end\', \'text_sha256\', \'branches\')} for c in selected]))\n    destination.mkdir(parents=True)\n    full = plan[\'full_scope\']\n    if full:\n        core.require(allowed_ids is not None and len(predictions) == 1200, \'Full submission requires official document allowlist\')\n        payload = core.canonical(predictions).encode()\n        (destination/\'submission.json\').write_bytes(payload)\n        with zipfile.ZipFile(destination/\'submission.zip\', \'w\', zipfile.ZIP_DEFLATED) as archive:\n            archive.writestr(\'submission.json\', payload)\n    else:\n        core.write(destination/\'PILOT_PREDICTIONS.json\', predictions)\n    with (destination/\'EVIDENCE_PROVENANCE.jsonl\').open(\'w\', encoding=\'utf-8\') as stream:\n        for row in provenance:\n            stream.write(core.canonical(row)+\'\\n\')\n    report = dict(state=\'full_submission_exported\' if full else \'pilot_only_not_for_submission\',\n        plan_signature=plan[\'signature\'], queries=len(predictions), source_batches=len(plan[\'batch_indices\']),\n        original_evidence_verified=True, official_ids_verified=allowed_ids is not None,\n        relevance_measured=False, official_lcs_measured=False,\n        documents=sum(len(p[\'relevant_docs\']) for p in predictions), chunks=sum(len(p[\'relevant_chunks\']) for p in predictions))\n    if full:\n        report[\'submission_zip_sha256\'] = core.file_hash(destination/\'submission.zip\')\n        report[\'submission_json_sha256\'] = core.file_hash(destination/\'submission.json\')\n    core.write(destination/\'REPORT.json\', report)\n    return report\n'}
SUPPORT = LOCAL / 'code'
SUPPORT.mkdir(exist_ok=True)
for filename, source in SUPPORT_FILES.items():
    module_name = filename[:-3]
    path = SUPPORT / filename
    if module_name in sys.modules:
        assert Path(sys.modules[module_name].__file__).read_text(encoding='utf-8') == source, 'Restart runtime: support code changed'
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(SUPPORT))
import corpus_multiretrieval as pipeline
from corpus_multiretrieval import core
import pyarrow.parquet as pq

def official_file(path, basename):
    if path.is_file():
        return path
    candidates = [ROOT/'Data/official/ViBioMIR'/basename, ROOT/'query'/basename]
    found = next((p for p in candidates if p.is_file()), None)
    if found:
        return found
    assert DATA_ZIP.is_file(), f'Missing {basename}; set its path or provide Data.zip'
    with zipfile.ZipFile(DATA_ZIP) as archive:
        matches = [n for n in archive.namelist() if Path(n).name == basename and 'official' in Path(n).parts]
        assert len(matches) == 1, f'Ambiguous/missing official file: {matches}'
        target = LOCAL / basename
        with archive.open(matches[0]) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    return target

QUERY_FILE = official_file(QUERY_FILE, 'query.parquet')
assert core.file_hash(QUERY_FILE) == core.EXPECTED_QUERY_SHA256, 'Wrong official query snapshot'
QUERIES = pq.read_table(QUERY_FILE, columns=['id', 'query']).to_pylist()

@contextmanager
def progress(label):
    finished = threading.Event()
    started = time.monotonic()
    def heartbeat():
        while not finished.wait(30):
            print(f'{label}: {(time.monotonic()-started)/60:.1f} minutes', flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    print(label, flush=True)
    thread.start()
    try:
        yield
    finally:
        finished.set()
        thread.join()


In [4]:
# SAME configuration in every retrieval session.
# Pilot: 20 queries, first 4 source batches. NOT a competition submission.
# Full run: set BOTH limits to None in ALL sessions; it creates a different RUN.
QUERY_LIMIT = None
SOURCE_BATCH_LIMIT = None
TOP_K_PER_BRANCH = 500
RERANK_TOP_K = 400
PLAN = pipeline.make_plan(QUERY_FILE, QUERIES, QUERY_LIMIT, SOURCE_BATCH_LIMIT,
                          TOP_K_PER_BRANCH, RERANK_TOP_K)
RUN = RUN_PARENT / PLAN['signature']
RUN.mkdir(parents=True, exist_ok=True)
plan_path = RUN / 'PLAN.json'
if plan_path.exists():
    assert core.checked(core.read(plan_path)) == PLAN, 'Different existing plan'
else:
    core.write(plan_path, PLAN)
print('RUN =', RUN)
print('Full scope:', PLAN['full_scope'], 'Queries:', len(PLAN['queries']), 'Batches:', len(PLAN['batch_indices']))


RUN = /content/drive/MyDrive/ViBioMIR/corpus_retrieval_trials/corpus_four_branch_v1/0340bda2ce4924b0cde8a2a2d4fd0ad80a0156fd18ef9b24fd4705abc0eeebba
Full scope: True Queries: 1200 Batches: 1712


In [5]:
ROLE = 'qwen'  # bm25 | bge | qwen | e5
SHARD_COUNT = 1
SHARD_INDEX = 0
SAVE_EMBEDDINGS = True
STOP_AFTER_BATCHES = None  # pilot timing; then None and rerun ONLY worker cell below
assert ROLE in pipeline.BRANCHES
assert ROLE != 'bm25' or (SHARD_COUNT, SHARD_INDEX) == (1, 0)
with progress('Inspect source ZIP metadata'):
    SOURCE_PLAN, MEMBERS = pipeline.setup_source(STAGE8_BUNDLES, PLAN)
print('Assigned batches:', pipeline.assigned(PLAN, SHARD_INDEX, SHARD_COUNT))


Inspect source ZIP metadata
Inspected worker_0.zip; total unique batches: 285
Inspected worker_1.zip; total unique batches: 570
Inspect source ZIP metadata: 0.5 minutes
Inspected worker_2.zip; total unique batches: 856
Inspected worker_3.zip; total unique batches: 1141
Inspect source ZIP metadata: 1.0 minutes
Inspected worker_4.zip; total unique batches: 1427
Inspected worker_5.zip; total unique batches: 1712
Assigned batches: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135

In [ ]:
with progress(f'Retrieve {ROLE}'):
    if ROLE == 'bm25':
        RESULT = pipeline.retrieve_bm25(PLAN, RUN, SOURCE_PLAN, MEMBERS,
            LOCAL / PLAN['signature'] / 'bm25.sqlite3', STOP_AFTER_BATCHES)
    else:
        RESULT = pipeline.retrieve_dense(PLAN, RUN, ROLE, SOURCE_PLAN, MEMBERS,
            SHARD_INDEX, SHARD_COUNT, STOP_AFTER_BATCHES, SAVE_EMBEDDINGS)
print('State:', RESULT['state'], 'Completed source batches:', len(RESULT['completed_batches']))


Retrieve qwen


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Retrieve qwen: 0.5 minutes
Embedding runtime: {'model': 'Qwen/Qwen3-Embedding-0.6B', 'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'dtype': 'torch.bfloat16', 'attention': 'eager', 'padding': 'left', 'vocabulary': 151669}


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Retrieve qwen: 1.0 minutes
Retrieve qwen: 1.5 minutes
Retrieve qwen: 2.0 minutes


Sau khi cả 4 nhánh và mọi shard đều complete, mở notebook 2.
Paused là checkpoint thời gian, chưa phải hoàn tất. Đổi STOP_AFTER_BATCHES=None rồi chạy lại cell worker.
Không chạy hai phiên cùng ROLE/SHARD_INDEX. Thay SHARD_COUNT phải dùng CAMPAIGN mới.
